In [19]:
import os
import pandas as pd
from sqlalchemy import create_engine
import win32com.client as win32
import time  # Для измерения времени выполнения
import shutil
import re
from glob import glob
# from glob import glob
import gc
from datetime import timedelta

# Функция для форматирования времени в часы, минуты и секунды
def format_elapsed_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours)} часа(ов) {int(minutes)} минут(ы) {seconds:.2f} секунд"

# Функция для проверки, является ли файл скрытым (для Windows)
def is_hidden(file_path):
    try:
        # Получаем атрибуты файла
        file_attributes = os.stat(file_path).st_file_attributes
        # Проверяем, установлен ли флаг "скрытый"
        return file_attributes & 2 != 0  # 2 соответствует атрибуту "скрытый"
    except Exception:
        # Если возникла ошибка, считаем файл не скрытым
        return False

# Функция для форматирования даты в строковый формат 'YYYY-MM-DD'
def format_date_column(df, date_column):
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce').dt.strftime('%Y-%m-%d')
    return df

# Функция для подключения к SQL Server с аутентификацией Windows
def connect_to_sql(server, database):
    connection_string = (
        f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )
    engine = create_engine(connection_string)
    return engine

# Функция для обработки ошибок и замены их на null
def handle_errors(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(lambda x: None if isinstance(x, str) and x.strip() == '' else x)
    return df

In [20]:
FOLDER_PATH = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!")
FOLDER_PATH_FEATURES = r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям"
FOLDER_PATH_FOR_DB= os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям")
FOLDER_PATH_DUDL = os.path.normpath(r"\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ")

SQL_SERVER = "cl01sql"
SQL_DATABASE_DBREPORT = "DBReport"
SQL_DATABASE_DBPARTNERS = "DBPartners"

In [21]:
print("Начинаем собирать Базу Данных...")
start_all_time = time.time()
engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)

Начинаем собирать Базу Данных...


In [22]:
import pandas as pd
import glob
import os
import datetime
import pyodbc

# Папки
path_voronka = r"Z:\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!\ВЫГРУЗКА воронка Озон"
path_zatraty = r"Z:\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!\Затраты\Озон. Затраты из Аналитики"

# === 1. ВОРОНКА ===
df_voronka_list = []
files_voronka = glob.glob(os.path.join(path_voronka, "analytics_report_*.xlsx"))

for file in files_voronka:
    # достаём дату из имени файла
    fname = os.path.basename(file)
    try:
        report_date = str(datetime.datetime.strptime(fname.split("_")[2], "%Y-%m-%d").date() - datetime.timedelta(days=1))
    except Exception:
        continue

    df = pd.read_excel(file, engine='calamine')

    # Чистим "Позиция в поиске и каталоге" от запятых
    df["Позиция в поиске и каталоге"] = (
        df["Позиция в поиске и каталоге"]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )

    # df["Позиция в поиске и каталоге"] = pd.to_numeric(df["Позиция в поиске и каталоге"], errors="coerce")
    # print(df.head(20))
    # типы
    df = df.astype({
        "Артикул": "string",
        "Показы, всего": "Int64",
        "Показы на карточке товара": "Int64",
        "Показы в поиске и каталоге": "Int64",
        "Позиция в поиске и каталоге": "float64",
        "В корзину, всего": "Int64",
        "Заказано товаров": "Int64",
        "Отменено товаров": "Int64",
        "Доставлено товаров": "Int64",
        "Возвращено товаров": "Int64",
        "Заказано на сумму": "float64",
        "В корзину из карточки товара": "Int64"
    })

    df["Дата"] = report_date

    df["Выкупили ШТ"] = df["Заказано товаров"] - df["Отменено товаров"] - df["Возвращено товаров"]
    df["Артикул"] = df["Артикул"].astype(str).str.split("-").str[0]

    df_voronka_list.append(df)
df_voronka = pd.concat(df_voronka_list, ignore_index=True)

sum_cols_all = [
    "Показы, всего",
    "Показы на карточке товара",
    "Показы в поиске и каталоге",
    "Позиция в поиске и каталоге",          # суммируем, как в вашем примере
    "В корзину, всего",
    "Заказано товаров",
    "Отменено товаров",
    "Доставлено товаров",
    "Возвращено товаров",
    "Заказано на сумму",
    "В корзину из карточки товара",
    "В корзину из поиска или каталога",     # если есть в выгрузке
    "Выкупили ШТ"
]
sum_cols = [c for c in sum_cols_all if c in df_voronka.columns]
df_voronka.rename(columns={
                        "Артикул": "Артикул",
                        "Продажи, ₽": "Рекламные заказано на сумму",
                        "Показы": "Рекламные показы",
                        "Клики": "Рекламные показы на карточке товара",
                        "Заказы, шт": "Рекламные заказано товаров"
                    }, inplace=True, errors="ignore")
# 2) безопасно приводим эти метрики к числам (NaN -> 0 перед суммированием)
for c in sum_cols:
    df_voronka[c] = pd.to_numeric(df_voronka[c], errors="coerce").fillna(0)

# 3) нечисловые поля, которые логично брать первыми в группе
first_cols_all = ["Тип товара", "Товары", "Модель", "Ozon ID"]
first_cols = [c for c in first_cols_all if c in df_voronka.columns]

# 4) готовим словарь агрегаций
agg_dict = {c: "sum" for c in sum_cols}
agg_dict.update({c: "first" for c in first_cols})

# 5) группируем и получаем одну строку на (Дата, Артикул)
df_voronka = (
    df_voronka
    .groupby(["Дата", "Артикул"], as_index=False)
    .agg(agg_dict)
)

In [23]:
df_voronka.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Ozon ID'],
      dtype='object')

In [24]:
str(df_voronka['Дата'].min())

'2026-02-28'

In [25]:
# 4. Получить данные из файла "Справочник.xlsx"
try:
    print("Начинаем получать данные для Справочника...")
    start_time = time.time()  # Запускаем таймер
    file_path_reference = os.path.join(FOLDER_PATH, "Справочник.xlsx")

    if os.path.exists(file_path_reference):
        # Список столбцов, которые нужно взять из файла
        columns_to_read = [
            "Артикул", "Артикул OZ", "Наименование", "Коллекция",
            "Бренд", "Размер", "Сезон", "Направление", "Розничный отдел",
            "Модель", "Группа", "Бизнес-группа", "Техсегмент",
            "Байер", "Две последние коллекции", "Основной артикул", "Ответственный за группу", "Себестоимость с НДС",
            "Процент выкупа", "НДС", "Группа для отчетов"
        ]

        # Типы данных для столбцов
        column_dtypes = {
            "Артикул": str,
            "Артикул OZ": str,
            "Наименование": str,
            "Коллекция": str,
            "Размер": str,
            "Бренд": str,
            "Сезон": str,
            "Направление": str,
            "Розничный отдел": str,
            "Модель": str,
            "Группа": str,
            "Бизнес-группа": str,
            "Техсегмент": str,
            "Байер": str,
            "Две последние коллекции": str,
            "Основной артикул": str,
            "Ответственный за группу": str,
            "Себестоимость с НДС": float,
            "Процент выкупа": float,
            "НДС": int,
            "Группа для отчетов": str
        }

        # Чтение файла с указанием нужных столбцов и типов данных
        df_reference = pd.read_excel(
            file_path_reference,
            sheet_name="Выгрузка для справочника",
            engine="openpyxl",
            usecols=columns_to_read,
            dtype=column_dtypes
        )

        # Удаление дубликатов
        df_reference = df_reference.drop_duplicates()

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Справочник:")
        print(df_reference.head())

        # Сохраняем результат
        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Справочника успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Справочник.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Справочника: {e}")

Начинаем получать данные для Справочника...
Первые 5 строк таблицы Справочник:
    Артикул                          Наименование Размер Коллекция Бренд  \
0  W9001403  Полуботинки женские зимние KF26085-1     36    2026AW  kari   
1  W9001403  Полуботинки женские зимние KF26085-1     38    2026AW  kari   
2  W9001403  Полуботинки женские зимние KF26085-1     37    2026AW  kari   
3  W9001403  Полуботинки женские зимние KF26085-1     39    2026AW  kari   
4  W9001403  Полуботинки женские зимние KF26085-1     40    2026AW  kari   

  Сезон    Направление Розничный отдел     Модель Бизнес-группа  ...  \
0  зима  Женская обувь   Женская обувь  KF26085-1         Обувь  ...   
1  зима  Женская обувь   Женская обувь  KF26085-1         Обувь  ...   
2  зима  Женская обувь   Женская обувь  KF26085-1         Обувь  ...   
3  зима  Женская обувь   Женская обувь  KF26085-1         Обувь  ...   
4  зима  Женская обувь   Женская обувь  KF26085-1         Обувь  ...   

  Техсегмент          Байер Две

In [26]:
# 7. Создание таблицы "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу ВсегоРазмеров...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Артикул", "Размер"]
    for col in required_columns:
        if col not in df_reference.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_reference.")
            exit()

    # Очищаем столбец "Размер":
    # - Преобразуем в строковый формат
    # - Удаляем лишние пробелы
    # - Заменяем пустые строки на None
    df_reference["Размер"] = df_reference["Размер"].astype(str).str.strip().replace('', None)

    # Создаем DataFrame с количеством размеров для каждого артикула
    df_reference_unique = (
        df_reference
        .drop_duplicates(subset=["Артикул", "Размер"])  # Удаляем дубликаты Артикул-Размер
        .groupby("Артикул")["Размер"]  # Группируем по артикулу
        .apply(lambda sizes: len(sizes.dropna().unique()) if len(sizes.dropna()) > 0 else 1)  # Подсчитываем размеры
        .reset_index(name="Всего размеров")
    )

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВсегоРазмеров:")
    print(df_reference_unique.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВсегоРазмеров успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВсегоРазмеров: {e}")

Начинаем создавать таблицу ВсегоРазмеров...
Первые 5 строк таблицы ВсегоРазмеров:
    Артикул  Всего размеров
0  00001851               5
1  00001852               5
2  00001855               5
3  00001856               5
4  00001931               6
Таблица ВсегоРазмеров успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 32.16 секунд


In [27]:
# 6. Получить данные таблицы с SQL (РазмерыНаАгрегаторе)
try:
    print("Начинаем получать данные для РазмеровНаАгрегаторе...")
    start_time = time.time()  # Запускаем таймер
    query_sizes = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], COUNT(DISTINCT(a.[INVENTSIZEID])) AS [Колво размеров]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{str(df_voronka['Дата'].min())}'
        GROUP BY [dt], [itemid]
    """
    df_sizes = pd.read_sql(query_sizes, engine)
    df_sizes = format_date_column(df_sizes, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы РазмерыНаАгрегаторе:")
    print(df_sizes.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для РазмеровНаАгрегаторе: {e}")

Начинаем получать данные для РазмеровНаАгрегаторе...
Первые 5 строк таблицы РазмерыНаАгрегаторе:
         Дата   Артикул  Колво размеров
0  2026-02-28  00128845               4
1  2026-02-28  00146325               6
2  2026-02-28  00146326               6
3  2026-02-28  00206040               5
4  2026-02-28  00206110               6
Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 30.09 секунд


In [28]:
# 8. Связать "РазмерыНаАгрегаторе" с "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу Дистрибуция...")
    start_time = time.time()  # Запускаем таймер

    # Объединяем таблицы по полю "Артикул"
    df_distribution = pd.merge(df_sizes, df_reference_unique, on="Артикул", how="left")

        # Вычисляем дистрибуцию с проверкой на деление на ноль
    df_distribution["Дистрибуция"] = df_distribution.apply(
    lambda row: row["Колво размеров"] / row["Всего размеров"] if row["Всего размеров"] != 0 else 0,
    axis=1
    )

    # Оставляем только нужные столбцы
    df_distribution = df_distribution[["Дата", "Артикул", "Дистрибуция"]]

    # Форматирование даты
    df_distribution = format_date_column(df_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Дистрибуция:")
    print(df_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Дистрибуция успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Дистрибуция: {e}")

Начинаем создавать таблицу Дистрибуция...
Первые 5 строк таблицы Дистрибуция:
         Дата   Артикул  Дистрибуция
0  2026-02-28  00128845     1.000000
1  2026-02-28  00146325     1.000000
2  2026-02-28  00146326     1.000000
3  2026-02-28  00206040     0.833333
4  2026-02-28  00206110     1.000000
Таблица Дистрибуция успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 19.59 секунд


In [29]:
# 2. Получить данные таблицы с SQL (Остатки)
try:
    print("Начинаем получать данные для Остатков...")
    start_time = time.time()  # Запускаем таймер
    query_stock = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], SUM(a.[free_to_sell_amount]) AS [Остаток Агрегатора]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{str(df_voronka['Дата'].min())}'
        GROUP BY [dt], [itemid]
    """
    df_stock = pd.read_sql(query_stock, engine)
    df_stock = format_date_column(df_stock, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатков:")
    print(df_stock.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для Остатков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для Остатков: {e}")

Начинаем получать данные для Остатков...
Первые 5 строк таблицы Остатков:
         Дата   Артикул  Остаток Агрегатора
0  2026-02-28  00128845                 451
1  2026-02-28  00146325                 225
2  2026-02-28  00146326                 231
3  2026-02-28  00206040                  58
4  2026-02-28  00206110                 307
Данные для Остатков успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 28.37 секунд


In [30]:
# 9. Связать "Остатки" с "Дистрибуция"
try:
    print("Начинаем создавать таблицу Остатки с дистрибуцией...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution = pd.merge(df_stock, df_distribution, on=["Дата", "Артикул"], how="left")
    df_stock_with_distribution = format_date_column(df_stock_with_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатки с дистрибуцией:")
    print(df_stock_with_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Остатки с дистрибуцией успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Остатки с дистрибуцией: {e}")

Начинаем создавать таблицу Остатки с дистрибуцией...
Первые 5 строк таблицы Остатки с дистрибуцией:
         Дата   Артикул  Остаток Агрегатора  Дистрибуция
0  2026-02-28  00128845                 451     1.000000
1  2026-02-28  00146325                 225     1.000000
2  2026-02-28  00146326                 231     1.000000
3  2026-02-28  00206040                  58     0.833333
4  2026-02-28  00206110                 307     1.000000
Таблица Остатки с дистрибуцией успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 2.83 секунд


In [31]:
del df_stock

In [32]:
df_voronka[df_voronka.duplicated(subset=["Артикул", "Дата", "Ozon ID"], keep=False)].sort_values(by='Артикул')

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,В корзину из поиска или каталога,Выкупили ШТ,Тип товара,Товары,Модель,Ozon ID


In [33]:
# === 2. ЗАТРАТЫ ===
df_zatraty_list = []
files_zatraty = glob.glob(os.path.join(path_zatraty, "*.csv"))

for f in files_zatraty:
    # дата из имени файла
    fname = os.path.basename(f).replace(".csv", "")
    file_date = pd.to_datetime(fname, dayfirst=True, errors="coerce")

    # читаем csv
    df = pd.read_csv(f, sep=";", skiprows=2)

    # чистим названия колонок
    df.columns = [c.replace(".csv","") if ".csv" in c else c for c in df.columns]

    # вставляем дату из имени файла
    df["Дата"] = file_date

    # переименования
    df = df.rename(columns={
        "Тип продвижения": "ТипАктивности",
        "Расход, ₽, с НДС": "Расход, ₽"
    })

    df_zatraty_list.append(df)

df_zatraty = pd.concat(df_zatraty_list, ignore_index=True)

In [34]:
df_zatraty

,SKU,ТипАктивности,ID кампании,"Расход, ₽","ДРР, %","Продажи, ₽","Заказы, шт","CTR, %",Показы,Клики,В корзину,"Конверсия в корзину, %","Затраты на заказ, ₽","Стоимость клика, ₽",Дата
0,1857636566,Оплата за клик,18836810,"1331,93",NaN,"0,00",0,"3,04",6280,191,11,"5,76",NaN,"6,97",2026-03-01
1,693052852,Оплата за клик,18842776,"167,48",NaN,"0,00",0,"3,54",1188,42,2,"4,76",NaN,"3,99",2026-03-01
2,1884357021,Оплата за клик,18842267,"1132,61",NaN,"0,00",0,"2,76",5879,162,11,"6,79",NaN,"6,99",2026-03-01
3,1400496303,Оплата за клик,18837211,"575,36",NaN,"0,00",0,"0,47",16569,78,2,"2,56",NaN,"7,38",2026-03-01
4,1645388294,Оплата за клик,18838672,"1169,58","69,10","1693,00",1,"4,36",6649,290,24,"8,28","1169,58","4,03",2026-03-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479140,1234929186,Оплата за клик,18842529,"0,00","0,00","0,00",0,"0,00",0,0,0,"0,00","0,00","0,00",2026-03-31
479141,1627854403,Оплата за клик,18839580,"0,00","0,00","0,00",0,"0,00",0,0,0,"0,00","0,00","0,00",2026-03-31
479142,178868001,Оплата за клик,18836087,"0,00","0,00","0,00",0,"0,00",0,0,0,"0,00","0,00","0,00",2026-03-31
479143,1403488144,Оплата за клик,23383677,"0,00","0,00","0,00",0,"0,00",0,0,0,"0,00","0,00","0,00",2026-03-31


In [35]:
# df_zatraty['ТипАктивности']
df_zatraty.loc[df_zatraty['ТипАктивности'] == 'Оплата за заказ: выбранные товары', 'ТипАктивности'] = 'Оплата за клик'

In [36]:
df_zatraty['ТипАктивности'].value_counts()

ТипАктивности
Оплата за клик    479145
Name: count, dtype: int64

In [37]:
df_zatraty[df_zatraty['Дата'] == '2026-04-22']['ТипАктивности'].unique()

array(['Оплата за клик'], dtype=object)

In [38]:
df_zatraty[df_zatraty['Дата'] == '2026-04-24']['ТипАктивности'].unique()

array(['Оплата за клик'], dtype=object)

In [111]:
df_zatraty[df_zatraty['Дата'] == '2026-04-25']['Показы']

407711    7519,00
407712    1568,00
407713    9388,00
407714    6007,00
407715     937,00
           ...   
414080       0,00
414081       0,00
414082       0,00
414083       0,00
414084       0,00
Name: Показы, Length: 6374, dtype: object

In [39]:
# === 3. SQL ЦЕНЫ ===
sql = f"""
select 'OZ' as AGREGATOR, DT, ITEMID, PRICE
from [DBPartners].[dbo].[WblmRepPriceDiscountOzReport]
where dt >= '{str(df_voronka['Дата'].min())}'
"""
df_prices = pd.read_sql(sql, engine)

df_prices = df_prices.groupby(["DT", "ITEMID"], as_index=False).agg({"PRICE": "max"})
df_prices = df_prices.rename(columns={"DT": "Дата", "ITEMID": "Артикул", "PRICE": "Цена"})

In [40]:
df_reference = df_reference.drop_duplicates(subset=["Артикул"])
df_reference = df_reference[["Артикул", "Бизнес-группа", "Направление", "Розничный отдел", "Группа", "Модель", "Бренд", "Коллекция", "Сезон", "Себестоимость с НДС", "Процент выкупа", "Две последние коллекции", 'Артикул OZ', 'Наименование', 'Техсегмент', 'Байер', 'Основной артикул', 'НДС', 'Ответственный за группу', 'Группа для отчетов']]
df_reference

,Артикул,Бизнес-группа,Направление,Розничный отдел,Группа,Модель,Бренд,Коллекция,Сезон,Себестоимость с НДС,Процент выкупа,Две последние коллекции,Артикул OZ,Наименование,Техсегмент,Байер,Основной артикул,НДС,Ответственный за группу,Группа для отчетов
0,W9001403,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,KF26085-1,kari,2026AW,зима,NaN,0.940870,2026AW,NaN,Полуботинки женские зимние KF26085-1,flat (L),Коновалова А.,W9001403,20,Гусева Дарья,Обувь
6,W9001404,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,KF26085-1A,kari,2026AW,зима,NaN,0.940870,2026AW,NaN,Полуботинки женские зимние KF26085-1A,flat (L),Коновалова А.,W9001404,20,Гусева Дарья,Обувь
12,W9001406,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,KF26085-2,kari,2026AW,зима,NaN,0.940870,2026AW,NaN,Полуботинки женские зимние KF26085-2,flat (L),Коновалова А.,W9001406,20,Гусева Дарья,Обувь
18,W9001407,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,KF26085-2A,kari,2026AW,зима,NaN,0.940870,2026AW,NaN,Полуботинки женские зимние KF26085-2A,flat (L),Коновалова А.,W9001407,20,Гусева Дарья,Обувь
23,W9001405,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,KF26085-1B,kari,2026AW,зима,NaN,0.940870,2026AW,NaN,Полуботинки женские зимние KF26085-1B,flat (L),Коновалова А.,W9001405,20,Гусева Дарья,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
728545,ya106010,Принадлежности для спорта и активного отдыха,Летний спорт,Велосипеды,ya1 Аксессуары для велосипедов,K10998,kari,2026SS,лето,85.6958,0.929116,"2025SS,2026SS",NaN,Корзина для велосипеда фуксия K10998,NaN,Бордачук Е. Спорт,ya106010,20,Шляпин Алексей,NaN
728546,ya108000,Принадлежности для спорта и активного отдыха,Летний спорт,Велосипеды,ya1 Аксессуары для велосипедов,JKPB254,kari,2025SS,лето,157.8034,0.929116,2025SS,NaN,"Ручка-толкатель для велосипеда 12""-16"" JKPB254",NaN,Бордачук Е. Спорт,ya108000,20,Шляпин Алексей,NaN
728547,ya102020,Принадлежности для спорта и активного отдыха,Летний спорт,Велосипеды,ya1 Аксессуары для велосипедов,K7379,KariKids,2023SS,лето,56.0560,0.929116,"2022SS,2023SS",NaN,Корзина для велосипеда черно-белая K7379,NaN,Бордачук Е. Спорт,ya102020,20,Шляпин Алексей,NaN
728548,ya106020,Принадлежности для спорта и активного отдыха,Летний спорт,Велосипеды,ya1 Аксессуары для велосипедов,K10997,Kari KIDS,2024SS,лето,79.8441,0.929116,2024SS,NaN,Корзина для велосипеда черная K10997,NaN,Бордачук Е. Спорт,ya106020,20,Шляпин Алексей,NaN


In [41]:
df_zatraty['SKU']

0         1857636566
1          693052852
2         1884357021
3         1400496303
4         1645388294
             ...    
479140    1234929186
479141    1627854403
479142     178868001
479143    1403488144
479144    1134313629
Name: SKU, Length: 479145, dtype: int64

In [42]:
import pandas as pd
import numpy as np

NBSP = '\u00A0'

# ---- утилиты ----
def norm_date(s: pd.Series) -> pd.Series:
    d = pd.to_datetime(s, errors='coerce')
    try:
        d = d.dt.tz_localize(None)
    except Exception:
        pass
    return d.dt.normalize()

def norm_code(s: pd.Series) -> pd.Series:
    return (s.astype(str)
              .str.replace(NBSP, '', regex=False)
              .str.strip()
              .str.upper())

def norm_code_alnum(s: pd.Series) -> pd.Series:
    # только A–Z и 0–9: удобно для OZ/SKU
    return norm_code(s).str.replace(r'[^0-9A-Z]+', '', regex=True)

def to_money(s: pd.Series) -> pd.Series:
    # «1 234,56 ₽», «1.234,56», «1234,56» → 1234.56
    ss = (s.astype(str)
            .str.replace(NBSP, '', regex=False)
            .str.replace(' ',  '', regex=False)
            .str.replace('₽',  '', regex=False))
    ss = ss.str.replace(r'(?<=\d)\.(?=\d{3}(?:\D|$))', '', regex=True)  # 1.234,56 → 1234,56
    ss = ss.str.replace(',', '.', regex=False)
    ss = ss.str.replace(r'[^0-9\.\-\+eE]', '', regex=True)
    return pd.to_numeric(ss, errors='coerce')

# ---- основная функция: тип берём ТОЛЬКО из затрат, без изменений ----
def merge_voronka_costs_preserve_impressions(
    df_voronka: pd.DataFrame,
    df_costs: pd.DataFrame,
    df_prices: pd.DataFrame | None = None,
    *,
    left_key_candidates=('Ozon ID','OZON ID','OZON_ID','Артикул OZ','Артикул'),
    right_key='SKU',
    spend_candidates=('Расход, ₽','Расход, руб','Расход, Р','Расход'),
    type_col_candidates=('ТипАктивности','Тип активности','Раздел', 'Тип продвижения'),
    preserve_cols=('Показы, всего',),   # инварианты по левым метрикам
    add_tail=True                        # добавлять «хвост» неприсоединившихся затрат
) -> pd.DataFrame:

    # --- копии + даты ---
    df_v = df_voronka.copy()
    df_z = df_costs.copy()
    df_p = None if df_prices is None else df_prices.copy()

    for dframe in (df_v, df_z) + ((df_p,) if df_p is not None else ()):
        if dframe is not None and 'Дата' in dframe.columns:
            dframe['Дата'] = norm_date(dframe['Дата'])

    # --- контроль инвариантов для левых метрик ---
    _num = lambda s: pd.to_numeric(s, errors='coerce')
    baseline = {c: (_num(df_v[c]).sum() if c in df_v.columns else None) for c in preserve_cols}

    # --- выбор ключей слева/справа ---
    left_key = next((c for c in left_key_candidates if c in df_v.columns), None)
    if left_key is None:
        raise KeyError(f"Во воронке нет ни одного ключа из {left_key_candidates}")

    if right_key not in df_z.columns:
        raise KeyError(f"В затратах нет колонки {right_key}")

    spend_col = next((c for c in spend_candidates if c in df_z.columns), None)
    if spend_col is None:
        raise KeyError(f"В затратах нет денежной колонки из {spend_candidates}")

    # --- подготовка затрат: ключ и деньги ---
    df_z[right_key] = norm_code(df_z[right_key])
    df_z['__KEY__'] = norm_code_alnum(df_z[right_key])
    df_z[spend_col] = to_money(df_z[spend_col]).astype('float64')

    # правый столбец типа (берём как есть, БЕЗ нормализаций)
    right_type_col = next((c for c in type_col_candidates if c in df_z.columns), None)

    # дополнительные правые метрики, если есть
    extra_metrics = [c for c in ['Показы','Клики','Заказы, шт','Продажи, ₽'] if c in df_z.columns]
    z_metrics = [spend_col] + extra_metrics

    # агрегирование по (Дата, __KEY__) + тип из затрат как first (как есть)
    if right_type_col is not None:
        agg_dict = {m: 'sum' for m in z_metrics}
        agg_dict[right_type_col] = 'first'  # важное: тип из затрат, без изменений
        z_agg_full = (df_z.groupby(['Дата','__KEY__'], as_index=False)
                        .agg(agg_dict))
        # переименуем правый тип в целевое имя
        if right_type_col != 'ТипАктивности':
            z_agg_full = z_agg_full.rename(columns={right_type_col: 'ТипАктивности'})
    else:
        z_agg_full = (df_z.groupby(['Дата','__KEY__'], as_index=False)[z_metrics]
                        .sum(min_count=1))

    total_costs = z_agg_full[spend_col].sum()

    # --- левый нормализованный ключ ---
    df_v['__KEY__'] = norm_code_alnum(norm_code(df_v[left_key]))
    df_v['IS_TAIL'] = 0

    # --- основной LEFT m:1 join ---
    df = df_v.merge(z_agg_full, on=['Дата','__KEY__'], how='left', validate='m:1')

    # если слева уже был 'ТипАктивности', мы его полностью ЗАМЕНЯЕМ на тип из затрат
    if 'ТипАктивности' in df.columns and right_type_col is None:
        # в затратах типа нет — тогда оставим как было слева
        pass
    else:
        # гарантируем, что колонка называется ровно 'ТипАктивности' и она пришла из затрат
        if right_type_col is not None and 'ТипАктивности' not in df.columns:
            # если правый тип имел другое имя и мы переименовали выше — уже ок
            # здесь просто убедимся, что колонка есть
            pass
        # Если и слева, и справа есть 'ТипАктивности' (редкий кейс),
        # берем ровно правую версию: удалим левую и переименуем правую.
        if right_type_col is not None and 'ТипАктивности_x' in df.columns and 'ТипАктивности_y' in df.columns:
            # _x — из воронки, _y — из затрат; оставляем _y
            df.drop(columns=['ТипАктивности_x'], inplace=True, errors='ignore')
            df.rename(columns={'ТипАктивности_y': 'ТипАктивности'}, inplace=True)

    # --- инварианты по левым метрикам ---
    for c in preserve_cols:
        if c in df.columns and baseline[c] is not None:
            after = _num(df[c]).sum()
            msg = "[OK]" if np.isclose(after, baseline[c], rtol=1e-9, atol=1e-6) else "[WARN]"
            print(f"{msg} Инвариант '{c}': до={baseline[c]:,.2f} | после={after:,.2f}")

    # --- добавляем «хвост» из затрат (без изменений типов!) ---
    tail_rows = 0
    if add_tail:
        left_keys = df[['Дата','__KEY__']].drop_duplicates()
        right_cols_only = [c for c in z_agg_full.columns if c not in ('Дата','__KEY__')]
        miss = (z_agg_full.merge(left_keys, on=['Дата','__KEY__'], how='left', indicator=True)
                          .loc[lambda x: x['_merge']=='left_only', ['Дата','__KEY__'] + right_cols_only])
        if not miss.empty:
            tail = miss.copy()
            # каркас: заполним недостающие левые поля NaN
            for col in df.columns:
                if col not in tail.columns:
                    tail[col] = np.nan
            tail['IS_TAIL'] = 1
            tail[left_key] = tail['__KEY__']
            tail = tail[df.columns]
            df = pd.concat([df, tail], ignore_index=True)
            tail_rows = len(tail)

    # --- прайс (m:1), если нужен ---
    if df_p is not None and {'Дата','Артикул'}.issubset(df.columns) and {'Дата','Артикул'}.issubset(df_p.columns):
        df_p['Артикул'] = norm_code(df_p['Артикул'])
        df = df.merge(df_p, on=['Дата','Артикул'], how='left', validate='m:1')

    # --- финальные сверки по расходам ---
    final_costs = pd.to_numeric(df[spend_col], errors='coerce').fillna(0).sum()
    delta = final_costs - total_costs
    print(f"[CHECK] Расходы: в файле = {total_costs:,.2f} ₽ | в результате = {final_costs:,.2f} ₽ | Δ = {delta:,.2f} ₽")
    if tail_rows:
        print(f"[INFO] Добавлен хвост (IS_TAIL=1): {tail_rows} строк")

    # --- уборка служебных ---
    df.drop(columns=['__KEY__','IS_TAIL'], inplace=True, errors='ignore')

    # --- опциональное переименование правых метрик в «рекламные …» ---
    df.rename(columns={
        "Продажи, ₽": "Рекламные заказано на сумму",
        "Показы": "Рекламные показы",
        "Клики": "Рекламные показы на карточке товара",
        "Заказы, шт": "Рекламные заказано товаров"
    }, inplace=True, errors="ignore")

    return df


# пример вызова
df_funnel = merge_voronka_costs_preserve_impressions(
    df_voronka=df_voronka,     # воронка (лево)
    df_costs=df_zatraty,       # затраты (право)
    df_prices=df_prices,       # опционально
    preserve_cols=('Показы, всего',),
    add_tail=False
)

[OK] Инвариант 'Показы, всего': до=5,484,211,330.00 | после=5,484,211,330.00
[CHECK] Расходы: в файле = 276,156,517.86 ₽ | в результате = 274,030,981.94 ₽ | Δ = -2,125,535.92 ₽


In [113]:
df_funnel[df_funnel['Дата'] == '2026-04-25']

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Товары,Модель,Артикул OZ,"Расход, ₽",Рекламные показы,Рекламные показы на карточке товара,Рекламные заказано товаров,Рекламные заказано на сумму,ТипАктивности,Цена
2921454,2026-04-25,00006050,1,0,0,0.00,0,0,0,0,...,Балетки T.TACCARDI,Балетки T.TACCARDI,149390940,0.0,NaN,NaN,NaN,NaN,Органика,628.0
2921455,2026-04-25,00006060,2,0,0,0.00,0,0,0,0,...,Балетки T.TACCARDI,Балетки T.TACCARDI,149390995,0.0,NaN,NaN,NaN,NaN,Органика,799.0
2921456,2026-04-25,00006080,2,0,0,0.00,0,0,0,0,...,Балетки T.TACCARDI,Балетки T.TACCARDI,149390920,0.0,NaN,NaN,NaN,NaN,Органика,690.0
2921457,2026-04-25,00006100,1,0,0,0.00,0,0,0,0,...,Балетки T.TACCARDI,Балетки T.TACCARDI,149393850,0.0,NaN,NaN,NaN,NaN,Органика,495.0
2921458,2026-04-25,00006110,1,0,0,0.00,0,0,0,0,...,Балетки T.TACCARDI,Балетки T.TACCARDI,149393771,0.0,NaN,NaN,NaN,NaN,Органика,495.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2971891,2026-04-25,Y9808110,205,12,64,856.31,2,0,0,0,...,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1997932756,0.0,NaN,NaN,NaN,NaN,Органика,NaN
2971892,2026-04-25,Y9808120,215,11,73,867.20,3,0,0,0,...,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1997933206,0.0,NaN,NaN,NaN,NaN,Органика,NaN
2971893,2026-04-25,Y9808130,111,13,37,524.80,0,0,1,0,...,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1934644792,0.0,NaN,NaN,NaN,NaN,Органика,NaN
2971894,2026-04-25,Y9808140,109,7,52,525.95,0,0,0,1,...,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1950208615,0.0,NaN,NaN,NaN,NaN,Органика,NaN


In [43]:
df_voronka.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Ozon ID'],
      dtype='object')

In [44]:
funnel_columns = df_funnel.columns

In [45]:
# Путь до папки
folder_path_weeks = os.path.join(FOLDER_PATH, "Затраты", "Озон. Затраты из Аналитики New Format")

# Собираем все .xlsx файлы
files = glob.glob(os.path.join(folder_path_weeks, "*.xlsx"))

df_list = []
df_list_union = []
for file in files:
    # --- достаём дату из названия файла ---
    filename = os.path.basename(file)  # например: "Аналитика продвижения_16.09.2025.xlsx"
    date_str = filename.split("_")[-1].replace(".xlsx", "")  # "16.09.2025"
    date_parsed = (pd.to_datetime(date_str, format="%d.%m.%Y") - timedelta(days=1)).strftime("%Y-%m-%d")

    # читаем, пропуская первую строку
    df_tmp = pd.read_excel(file, engine='calamine', skiprows=1)
    df_tmp_union = pd.read_excel(file, sheet_name='Union', engine='calamine', skiprows=1)

    # оставляем только нужные колонки
    cols_keep = ["SKU", "ID кампании", "Инструмент", "Место размещения"]
    cols_keep_union = ["SKU в продвижении", "SKU из объединенной карточки", "Продажи, ₽", "Заказы, шт"]
    df_tmp = df_tmp[cols_keep]
    df_tmp_union = df_tmp_union[cols_keep_union]

    # добавляем колонку "Дата"
    df_tmp["Дата"] = date_parsed
    df_tmp_union["Дата"] = date_parsed

    df_list.append(df_tmp)
    df_list_union.append(df_tmp_union)

# объединяем все файлы
df_all = pd.concat(df_list, ignore_index=True)
df_all_union = pd.concat(df_list_union, ignore_index=True)
df_all.rename(columns={'SKU': "Артикул OZ"},inplace=True)
df_all_union.rename(columns={'SKU в продвижении': "Артикул OZ"},inplace=True)

# Функция для строгой нормализации
def normalize_key_column(s: pd.Series, col_name: str) -> pd.Series:
    """Строгая нормализация с логированием"""
    original = s.copy()
    
    result = (s.astype(str)
              .str.replace('\u00A0', '', regex=False)  # неразрывный пробел
              .str.replace(' ', '', regex=False)        # обычный пробел
              .str.replace('\t', '', regex=False)       # табуляция
              .str.strip()
              .str.upper())
    
    changed = (original.astype(str) != result).sum()
    print(f"  Колонка '{col_name}': изменено {changed} значений")
    
    return result

# Применяем ко ВСЕМ ключевым колонкам
print("Нормализация df_zatraty:")
df_zatraty['Дата'] = pd.to_datetime(df_zatraty['Дата'], errors='coerce').dt.normalize()
df_zatraty['SKU'] = normalize_key_column(df_zatraty['SKU'], 'SKU в затратах')
df_zatraty['ID кампании'] = normalize_key_column(df_zatraty['ID кампании'], 'ID в затратах')

print("\nНормализация df_all:")
df_all['Дата'] = pd.to_datetime(df_all['Дата'], errors='coerce').dt.normalize()
df_all['Артикул OZ'] = normalize_key_column(df_all['Артикул OZ'], 'SKU в df_all')
df_all['ID кампании'] = normalize_key_column(df_all['ID кампании'], 'ID в df_all')

# # Нормализуем ключи (точно так же, как в merge_voronka_costs_preserve_impressions)
# df_all['Дата'] = pd.to_datetime(df_all['Дата'], errors='coerce').dt.normalize()
df_all['Артикул OZ'] = (df_all['Артикул OZ'].astype(str)
                 .str.replace('\u00A0', '', regex=False)
                 .str.strip()
                 .str.upper())
df_all_union['Артикул OZ'] = (df_all_union['Артикул OZ'].astype(str)
                 .str.replace('\u00A0', '', regex=False)
                 .str.strip()
                 .str.upper())
df_zatraty['SKU'] = (df_zatraty['SKU'].astype(str)
                 .str.replace('\u00A0', '', regex=False)
                 .str.strip()
                 .str.upper())

# # ===== ШАГ 2: Обогащаем df_zatraty информацией из df_all =====
# # ВАЖНО: делаем это ДО вызова merge_voronka_costs_preserve_impressions

# Нормализуем дату в df_zatraty
df_zatraty['Дата'] = pd.to_datetime(df_zatraty['Дата'], errors='coerce').dt.normalize()
df_all['Дата'] = pd.to_datetime(df_all['Дата'], errors='coerce').dt.normalize()

# Присоединяем инструменты по (SKU, ID кампании, Дата)
df_zatraty_enriched = df_zatraty.rename(columns={'SKU': "Артикул OZ"}).merge(
    df_all[['Дата', 'Артикул OZ', 'ID кампании', 'Инструмент', 'Место размещения']],
    on=['Дата', 'Артикул OZ', 'ID кампании'],
    how='left',
    validate='m:1'  # каждая строка затрат должна найти максимум одну строку в df_all
)

# ===== ШАГ 3: Создаём ТипАктивности В ЗАТРАТАХ (до слияния с воронкой) =====
df_zatraty_enriched['ТипАктивности'] = np.nan

# Трафареты
mask_trafarety = (
    (df_zatraty_enriched['Инструмент'] == 'Оплата за клик') &
    (df_zatraty_enriched['Место размещения'] == 'Поиск и рекомендации')
)
df_zatraty_enriched.loc[mask_trafarety, 'ТипАктивности'] = 'Трафареты'

# Вывод в топ
mask_top = (
    (df_zatraty_enriched['Инструмент'] == 'Оплата за клик') &
    (df_zatraty_enriched['Место размещения'] == 'Поиск')
)
df_zatraty_enriched.loc[mask_top, 'ТипАктивности'] = 'Вывод в топ'

# Органика (расход = 0)
spend_col = next((c for c in ['Расход, ₽','Расход, руб','Расход, Р','Расход'] 
                  if c in df_zatraty_enriched.columns), None)
if spend_col:
    df_zatraty_enriched[spend_col] = pd.to_numeric(
        df_zatraty_enriched[spend_col].astype(str)
        .str.replace('\u00A0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace('₽', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce'
    )
    mask_organic = (df_zatraty_enriched[spend_col] == 0) | (df_zatraty_enriched[spend_col].isna())
    df_zatraty_enriched.loc[mask_organic, 'ТипАктивности'] = 'Органика'

# ===== ДИАГНОСТИКА 4: После merge с df_all =====
# df_zatraty_enriched = df_zatraty.merge(
#     df_all[['Дата', 'SKU', 'ID кампании', 'Инструмент', 'Место размещения']],
#     on=['Дата', 'SKU', 'ID кампании'],
#     how='left',
#     indicator=True  # добавляем индикатор
# )

# print("\n" + "=" * 80)
# print("4. После merge df_zatraty + df_all")
# print(df_zatraty_enriched['_merge'].value_counts())
# print(f"\nСтрок с заполненным 'Инструмент': {df_zatraty_enriched['Инструмент'].notna().sum():,}")
# print(f"Строк с пустым 'Инструмент': {df_zatraty_enriched['Инструмент'].isna().sum():,}")

# # Примеры строк, которые НЕ смержились
# not_merged = df_zatraty_enriched[df_zatraty_enriched['_merge'] == 'left_only'].head(10)
# print("\n10 примеров строк, которые НЕ нашли match в df_all:")
# print(not_merged[['Дата', 'SKU', 'ID кампании', 'Инструмент', '_merge']])

# df_zatraty_enriched.drop(columns=['_merge'], inplace=True)

# # Проверяем распределение ПЕРЕД слиянием
# print("Распределение ТипАктивности в затратах (до слияния с воронкой):")
# print(df_zatraty_enriched['ТипАктивности'].value_counts(dropna=False))

# # ===== ШАГ 4: Вызываем merge_voronka_costs_preserve_impressions =====
# df_funnel = merge_voronka_costs_preserve_impressions(
#     df_voronka=df_voronka,
#     df_costs=df_zatraty_enriched,  # используем обогащённые затраты
#     df_prices=df_prices,
#     preserve_cols=('Показы, всего',),
#     right_key='Артикул OZ',
#     add_tail=True,
#     type_col_candidates=('ТипАктивности',)  # указываем, что тип уже в затратах
# )

Нормализация df_zatraty:
  Колонка 'SKU в затратах': изменено 0 значений
  Колонка 'ID в затратах': изменено 0 значений

Нормализация df_all:
  Колонка 'SKU в df_all': изменено 0 значений
  Колонка 'ID в df_all': изменено 0 значений


C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_16564\2549437887.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Трафареты' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_zatraty_enriched.loc[mask_trafarety, 'ТипАктивности'] = 'Трафареты'


In [46]:
print("=" * 80)
print("ПРАВИЛЬНЫЙ МЕРЖ БЕЗ ЛИШНЕГО МАППИНГА")
print("=" * 80)

# ===== ШАГ 1: Подготовка воронки =====
print("\nШАГ 1: Подготовка воронки")
print("-" * 80)

df_voronka_clean = df_voronka.copy()

# Убедимся, что Ozon ID есть
if 'Ozon ID' not in df_voronka_clean.columns:
    raise ValueError("❌ В воронке нет колонки 'Ozon ID'!")

# Нормализуем Ozon ID - приводим к строке и убираем лишнее
df_voronka_clean['Ozon ID'] = (
    df_voronka_clean['Ozon ID']
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r'\.0$', '', regex=True)  # убираем .0 если есть
)

print(f"✓ Размер воронки: {len(df_voronka_clean):,} строк")
print(f"✓ Заполненность Ozon ID: {df_voronka_clean['Ozon ID'].notna().sum():,} ({100*df_voronka_clean['Ozon ID'].notna().sum()/len(df_voronka_clean):.1f}%)")
print(f"✓ Примеры Ozon ID: {df_voronka_clean['Ozon ID'].head(5).tolist()}")

# ===== ШАГ 2: Подготовка затрат =====
print("\nШАГ 2: Подготовка затрат")
print("-" * 80)

df_zatraty_clean = df_zatraty_enriched.copy()

# Нормализуем Артикул OZ
df_zatraty_clean['Артикул OZ'] = (
    df_zatraty_clean['Артикул OZ']
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r'\.0$', '', regex=True)
)

print(f"✓ Размер затрат: {len(df_zatraty_clean):,} строк")
print(f"✓ Уникальных Артикул OZ: {df_zatraty_clean['Артикул OZ'].nunique():,}")
print(f"✓ Примеры Артикул OZ: {df_zatraty_clean['Артикул OZ'].head(5).tolist()}")

# ===== ШАГ 3: Проверка пересечения =====
print("\nШАГ 3: Проверка пересечения ключей")
print("-" * 80)

voronka_ids = set(df_voronka_clean['Ozon ID'].dropna())
zatraty_ids = set(df_zatraty_clean['Артикул OZ'].dropna())

intersection = voronka_ids & zatraty_ids

print(f"Уникальных ID в воронке: {len(voronka_ids):,}")
print(f"Уникальных ID в затратах: {len(zatraty_ids):,}")
print(f"Пересечение: {len(intersection):,} ({100*len(intersection)/len(voronka_ids) if voronka_ids else 0:.1f}% от воронки)")

if len(intersection) == 0:
    print("\n⚠️ КРИТИЧЕСКАЯ ПРОБЛЕМА: НЕТ ПЕРЕСЕЧЕНИЙ!")
    print("Возможные причины:")
    print("  1. Разные периоды данных в воронке и затратах")
    print("  2. Разные площадки (один файл WB, другой Ozon)")
    print("  3. Проблема с ID - один Ozon ID, другой SKU")
    
    print("\nПримеры ID из воронки:")
    print(list(voronka_ids)[:10])
    print("\nПримеры ID из затрат:")
    print(list(zatraty_ids)[:10])
    
    raise ValueError("Невозможно продолжить без пересечений!")

# ===== ШАГ 4: Мерж через функцию =====
print("\nШАГ 4: Финальный мерж")
print("-" * 80)

# Переименуем Ozon ID -> Артикул OZ для единообразия
df_voronka_clean = df_voronka_clean.rename(columns={'Ozon ID': 'Артикул OZ'})

df_prices_exists = 'df_prices' in globals()

df_funnel_final = merge_voronka_costs_preserve_impressions(
    df_voronka=df_voronka_clean,
    df_costs=df_zatraty_clean,
    df_prices=df_prices if df_prices_exists else None,
    preserve_cols=('Показы, всего',),
    left_key_candidates=('Артикул OZ',),    # ← теперь одинаковое название!
    right_key='Артикул OZ',                 # ← и тут тоже!
    add_tail=False,
    type_col_candidates=('ТипАктивности',)
)

print("\n✅ Мерж завершён!")

# ===== ШАГ 5: Заполняем органику =====
print("\nШАГ 5: Заполнение органического трафика")
print("-" * 80)

spend_col = next((c for c in ['Расход, ₽','Расход, руб'] if c in df_funnel_final.columns), None)
shows_col = next((c for c in ['Показы, всего', 'Показы'] if c in df_funnel_final.columns), None)

if spend_col and shows_col:
    # Нормализуем колонки
    df_funnel_final[spend_col] = pd.to_numeric(df_funnel_final[spend_col], errors='coerce').fillna(0)
    df_funnel_final[shows_col] = pd.to_numeric(df_funnel_final[shows_col], errors='coerce').fillna(0)
    
    # Маска для органики: нет типа + расход=0 + есть показы
    mask_organic = (
        df_funnel_final['ТипАктивности'].isna() &
        (df_funnel_final[spend_col] == 0) &
        (df_funnel_final[shows_col] > 0)
    )
    
    organic_count_before = mask_organic.sum()
    df_funnel_final.loc[mask_organic, 'ТипАктивности'] = 'Органика'
    
    print(f"✓ Заполнено 'Органика' для {organic_count_before:,} строк")

# ===== ШАГ 6: Финальная статистика =====
print("\n" + "=" * 80)
print("ФИНАЛЬНАЯ СТАТИСТИКА")
print("=" * 80)

print(f"\nРазмер результата: {len(df_funnel_final):,} строк")

print("\nРаспределение ТипАктивности:")
type_counts = df_funnel_final['ТипАктивности'].value_counts(dropna=False)
print(type_counts)

nan_count = df_funnel_final['ТипАктивности'].isna().sum()
nan_pct = 100 * nan_count / len(df_funnel_final)
print(f"\nСтрок с NaN: {nan_count:,} ({nan_pct:.1f}%)")

if nan_pct < 20:
    print("\n✅ УСПЕХ! Доля NaN < 20%")
elif nan_pct < 50:
    print("\n⚠️ ЧАСТИЧНО РЕШЕНО: Доля NaN {:.1f}%".format(nan_pct))
else:
    print("\n❌ ПРОБЛЕМА: Доля NaN всё ещё {:.1f}%".format(nan_pct))

# Проверка расходов
if spend_col:
    total_spend = df_funnel_final[spend_col].sum()
    original_spend = df_zatraty_clean[spend_col].sum()
    print(f"\nСумма расходов:")
    print(f"  - В затратах: {original_spend:,.2f} ₽")
    print(f"  - В результате: {total_spend:,.2f} ₽")
    print(f"  - Разница: {total_spend - original_spend:,.2f} ₽")

print("\n" + "=" * 80)
print("ГОТОВО! Результат в переменной df_funnel_final")
print("=" * 80)

# Сохраняем в df_funnel для удобства
df_funnel = df_funnel_final
print("\n✓ Также сохранено в df_funnel")

ПРАВИЛЬНЫЙ МЕРЖ БЕЗ ЛИШНЕГО МАППИНГА

ШАГ 1: Подготовка воронки
--------------------------------------------------------------------------------
✓ Размер воронки: 3,075,806 строк
✓ Заполненность Ozon ID: 3,075,806 (100.0%)
✓ Примеры Ozon ID: ['149393764', '149390921', '149354561', '149372421', '149393771']

ШАГ 2: Подготовка затрат
--------------------------------------------------------------------------------
✓ Размер затрат: 479,145 строк
✓ Уникальных Артикул OZ: 18,080
✓ Примеры Артикул OZ: ['1857636566', '693052852', '1884357021', '1400496303', '1645388294']

ШАГ 3: Проверка пересечения ключей
--------------------------------------------------------------------------------
Уникальных ID в воронке: 210,605
Уникальных ID в затратах: 18,080
Пересечение: 15,669 (7.4% от воронки)

ШАГ 4: Финальный мерж
--------------------------------------------------------------------------------
[OK] Инвариант 'Показы, всего': до=5,484,211,330.00 | после=5,484,211,330.00
[CHECK] Расходы: в файле = 2

In [47]:
df_funnel['ТипАктивности'].value_counts()

ТипАктивности
Органика       2660295
Трафареты       412036
Вывод в топ         41
Name: count, dtype: int64

In [48]:
# print("=" * 80)
# print("ПЕРЕКЛАССИФИКАЦИЯ 'НЕИЗВЕСТНЫЙ ТИП'")
# print("=" * 80)

# # Проверяем наличие df_funnel
# if 'df_funnel' not in globals():
#     raise ValueError("❌ df_funnel не найден! Запустите сначала основной мерж.")

# df_funnel_fixed = df_funnel.copy()

# # Находим колонку расходов
# spend_col = next((c for c in ['Расход, ₽','Расход, руб','Расход'] if c in df_funnel_fixed.columns), None)

# if spend_col is None:
#     raise ValueError("❌ Не найдена колонка расходов!")

# # Нормализуем расходы
# df_funnel_fixed[spend_col] = pd.to_numeric(df_funnel_fixed[spend_col], errors='coerce').fillna(0)

# # ===== ВАРИАНТ 1: На основе расходов =====
# print("\nВАРИАНТ 1: Классификация на основе РАСХОДОВ")
# print("-" * 80)

# # Маска: Неизвестный тип + ненулевые расходы
# mask_unknown_with_spend = (
#     (df_funnel_fixed['ТипАктивности'] == 'Неизвестный тип') &
#     (df_funnel_fixed[spend_col] > 0)
# )

# unknown_count = (df_funnel_fixed['ТипАктивности'] == 'Неизвестный тип').sum()
# unknown_with_spend = mask_unknown_with_spend.sum()

# print(f"Всего 'Неизвестный тип': {unknown_count:,}")
# print(f"Из них с расходами > 0: {unknown_with_spend:,}")

# if unknown_with_spend > 0:
#     # По умолчанию считаем что это Трафареты
#     # (т.к. это самый частый тип рекламы на Озоне)
#     df_funnel_fixed.loc[mask_unknown_with_spend, 'ТипАктивности'] = 'Трафареты'
    
#     print(f"\n✓ Переклассифицировано {unknown_with_spend:,} строк в 'Трафареты'")
#     print("  (строки с расходами, но без информации об инструменте)")

# # ===== ВАРИАНТ 2: Проверка наличия инструмента и места =====
# print("\n\nВАРИАНТ 2: Классификация на основе ИНСТРУМЕНТА (если доступен)")
# print("-" * 80)

# if 'Инструмент' in df_funnel_fixed.columns and 'Место размещения' in df_funnel_fixed.columns:
#     # Для оставшихся "Неизвестный тип" с инструментом - классифицируем правильно
    
#     # Трафареты: Оплата за клик + Поиск и рекомендации
#     mask_trafarety = (
#         (df_funnel_fixed['ТипАктивности'] == 'Неизвестный тип') &
#         (df_funnel_fixed['Инструмент'] == 'Оплата за клик') &
#         (df_funnel_fixed['Место размещения'] == 'Поиск и рекомендации')
#     )
    
#     # Вывод в топ: Оплата за клик + Поиск
#     mask_top = (
#         (df_funnel_fixed['ТипАктивности'] == 'Неизвестный тип') &
#         (df_funnel_fixed['Инструмент'] == 'Оплата за клик') &
#         (df_funnel_fixed['Место размещения'] == 'Поиск')
#     )
    
#     traf_count = mask_trafarety.sum()
#     top_count = mask_top.sum()
    
#     if traf_count > 0:
#         df_funnel_fixed.loc[mask_trafarety, 'ТипАктивности'] = 'Трафареты'
#         print(f"✓ Найдено и переклассифицировано {traf_count:,} как 'Трафареты'")
    
#     if top_count > 0:
#         df_funnel_fixed.loc[mask_top, 'ТипАктивности'] = 'Вывод в топ'
#         print(f"✓ Найдено и переклассифицировано {top_count:,} как 'Вывод в топ'")
    
#     if traf_count == 0 and top_count == 0:
#         print("  Нет строк с достаточной информацией для точной классификации")
# else:
#     print("  Колонки 'Инструмент' или 'Место размещения' отсутствуют")
#     print("  Используется только Вариант 1 (на основе расходов)")

# # ===== ВАРИАНТ 3: Органика для нулевых расходов =====
# print("\n\nВАРИАНТ 3: Оставшиеся с нулевыми расходами → ОРГАНИКА")
# print("-" * 80)

# mask_unknown_zero = (
#     (df_funnel_fixed['ТипАктивности'] == 'Неизвестный тип') &
#     (df_funnel_fixed[spend_col] == 0)
# )

# zero_count = mask_unknown_zero.sum()

# if zero_count > 0:
#     df_funnel_fixed.loc[mask_unknown_zero, 'ТипАктивности'] = 'Органика'
#     print(f"✓ Переклассифицировано {zero_count:,} строк в 'Органика'")
#     print("  (строки без расходов)")

# # ===== ФИНАЛЬНАЯ СТАТИСТИКА =====
# print("\n" + "=" * 80)
# print("ФИНАЛЬНАЯ СТАТИСТИКА ПОСЛЕ ПЕРЕКЛАССИФИКАЦИИ")
# print("=" * 80)

# print(f"\nРазмер: {len(df_funnel_fixed):,} строк")

# print("\nРаспределение ТипАктивности:")
# type_counts = df_funnel_fixed['ТипАктивности'].value_counts(dropna=False)
# print(type_counts)

# nan_count = df_funnel_fixed['ТипАктивности'].isna().sum()
# unknown_remaining = (df_funnel_fixed['ТипАктивности'] == 'Неизвестный тип').sum()

# print(f"\nСтрок с NaN: {nan_count:,} ({100*nan_count/len(df_funnel_fixed):.1f}%)")
# print(f"Строк с 'Неизвестный тип': {unknown_remaining:,} ({100*unknown_remaining/len(df_funnel_fixed):.1f}%)")

# total_unclear = nan_count + unknown_remaining
# print(f"\nВСЕГО неопределённых: {total_unclear:,} ({100*total_unclear/len(df_funnel_fixed):.1f}%)")

# if total_unclear / len(df_funnel_fixed) < 0.20:
#     print("\n✅ ОТЛИЧНО! Менее 20% неопределённых строк")
# elif total_unclear / len(df_funnel_fixed) < 0.50:
#     print("\n⚠️ ПРИЕМЛЕМО: 20-50% неопределённых строк")
# else:
#     print("\n❌ ПРОБЛЕМА: Более 50% неопределённых строк")

# # Проверка расходов
# total_spend = df_funnel_fixed[spend_col].sum()
# print(f"\nОбщая сумма расходов: {total_spend:,.2f} ₽")

# print("\n" + "=" * 80)
# print("ГОТОВО! Результат сохранён в df_funnel_fixed")
# print("=" * 80)

# # Перезаписываем df_funnel
# df_funnel = df_funnel_fixed
# print("\n✓ Также обновлён df_funnel")

In [49]:
df_funnel['ТипАктивности'].value_counts()

ТипАктивности
Органика       2660295
Трафареты       412036
Вывод в топ         41
Name: count, dtype: int64

In [50]:
print("=" * 80)
print("УГЛУБЛЕННАЯ ДИАГНОСТИКА ПРОБЛЕМЫ")
print("=" * 80)

# ===== 1. Проверяем исходную воронку =====
print("\n1. ИСХОДНАЯ ВОРОНКА (df_voronka)")
print("-" * 80)

# Загружаем оригинальную воронку заново (без маппинга)
# ВНИМАНИЕ: Если df_voronka уже изменён, нужно перезапустить ячейки загрузки!

print(f"Размер: {len(df_voronka):,} строк")
print(f"Колонки: {df_voronka.columns.tolist()}")

# Проверяем наличие Ozon ID
if 'Ozon ID' in df_voronka.columns:
    print("\n✅ В воронке УЖЕ ЕСТЬ колонка 'Ozon ID'!")
    
    # Смотрим на заполненность
    ozon_id_filled = df_voronka['Ozon ID'].notna().sum()
    ozon_id_pct = 100 * ozon_id_filled / len(df_voronka)
    print(f"  - Заполнено: {ozon_id_filled:,} строк ({ozon_id_pct:.1f}%)")
    print(f"  - Пусто (NaN): {df_voronka['Ozon ID'].isna().sum():,} строк")
    
    # Примеры значений
    print(f"\nПримеры Ozon ID из воронки:")
    print(df_voronka['Ozon ID'].dropna().head(10).tolist())
else:
    print("\n❌ В воронке НЕТ колонки 'Ozon ID'")

# Проверяем Артикул
if 'Артикул' in df_voronka.columns:
    print(f"\nПримеры Артикул из воронки:")
    print(df_voronka['Артикул'].dropna().head(10).tolist())

# ===== 2. Проверяем затраты =====
print("\n" + "=" * 80)
print("2. ЗАТРАТЫ (df_zatraty_enriched)")
print("-" * 80)

print(f"Размер: {len(df_zatraty_enriched):,} строк")

if 'Артикул OZ' in df_zatraty_enriched.columns:
    print(f"\nПримеры Артикул OZ из затрат:")
    print(df_zatraty_enriched['Артикул OZ'].dropna().head(10).tolist())
    
    # Уникальные значения
    unique_oz = df_zatraty_enriched['Артикул OZ'].nunique()
    print(f"\nУникальных Артикул OZ в затратах: {unique_oz:,}")

# Проверяем ТипАктивности В ЗАТРАТАХ
if 'ТипАктивности' in df_zatraty_enriched.columns:
    print(f"\nРаспределение ТипАктивности В ЗАТРАТАХ:")
    print(df_zatraty_enriched['ТипАктивности'].value_counts(dropna=False))

# ===== 3. Сравниваем ключи =====
print("\n" + "=" * 80)
print("3. СРАВНЕНИЕ КЛЮЧЕЙ")
print("-" * 80)

# Проверяем совпадение по Ozon ID (если он есть)
if 'Ozon ID' in df_voronka.columns:
    print("\nСравнение Ozon ID из воронки и Артикул OZ из затрат:")
    
    # Нормализуем для сравнения
    voronka_ids = set(df_voronka['Ozon ID'].dropna().astype(str).str.strip().str.upper())
    zatraty_ids = set(df_zatraty_enriched['Артикул OZ'].dropna().astype(str).str.strip().str.upper())
    
    print(f"  - Уникальных ID в воронке: {len(voronka_ids):,}")
    print(f"  - Уникальных ID в затратах: {len(zatraty_ids):,}")
    
    intersection = voronka_ids & zatraty_ids
    print(f"  - Пересечение: {len(intersection):,} ({100*len(intersection)/len(voronka_ids) if voronka_ids else 0:.1f}%)")
    
    # Примеры НЕ совпавших
    only_voronka = voronka_ids - zatraty_ids
    only_zatraty = zatraty_ids - voronka_ids
    
    print(f"\n  - Только в воронке: {len(only_voronka):,}")
    if only_voronka:
        print(f"    Примеры: {list(only_voronka)[:5]}")
    
    print(f"  - Только в затратах: {len(only_zatraty):,}")
    if only_zatraty:
        print(f"    Примеры: {list(only_zatraty)[:5]}")

# ===== 4. Проверяем результат мержа =====
print("\n" + "=" * 80)
print("4. АНАЛИЗ РЕЗУЛЬТАТА МЕРЖА (df_funnel)")
print("-" * 80)

# Проверяем, какие строки не получили тип
nan_rows = df_funnel[df_funnel['ТипАктивности'].isna()]
print(f"\nСтрок с NaN: {len(nan_rows):,}")

# Есть ли у них расходы?
spend_col = next((c for c in ['Расход, ₽','Расход, руб'] if c in df_funnel.columns), None)
if spend_col:
    spend_in_nan = pd.to_numeric(nan_rows[spend_col], errors='coerce')
    
    print(f"\nАнализ NaN строк:")
    print(f"  - С ненулевыми расходами: {(spend_in_nan > 0).sum():,}")
    print(f"  - С нулевыми расходами: {(spend_in_nan == 0).sum():,}")
    print(f"  - С NaN расходами: {spend_in_nan.isna().sum():,}")

# Есть ли у них показы?
shows_col = next((c for c in ['Показы, всего', 'Показы'] if c in df_funnel.columns), None)
if shows_col:
    shows_in_nan = pd.to_numeric(nan_rows[shows_col], errors='coerce')
    
    print(f"\n  - С ненулевыми показами: {(shows_in_nan > 0).sum():,}")
    print(f"  - С нулевыми показами: {(shows_in_nan == 0).sum():,}")

# ===== 5. КЛЮЧЕВОЙ ВОПРОС: Что с органикой? =====
print("\n" + "=" * 80)
print("5. ОРГАНИЧЕСКИЙ ТРАФИК")
print("-" * 80)

# Строки с показами, но без расходов = органика
if spend_col and shows_col:
    df_funnel[spend_col] = pd.to_numeric(df_funnel[spend_col], errors='coerce').fillna(0)
    df_funnel[shows_col] = pd.to_numeric(df_funnel[shows_col], errors='coerce').fillna(0)
    
    organic_mask = (
        (df_funnel[spend_col] == 0) &
        (df_funnel[shows_col] > 0)
    )
    
    organic_count = organic_mask.sum()
    print(f"Потенциальных органических строк (расход=0, показы>0): {organic_count:,}")
    
    # Сколько из них имеют NaN в типе?
    organic_nan = (organic_mask & df_funnel['ТипАктивности'].isna()).sum()
    print(f"  - Из них с NaN в ТипАктивности: {organic_nan:,} ({100*organic_nan/organic_count if organic_count else 0:.1f}%)")
    
    # Примеры органических строк без типа
    if organic_nan > 0:
        print("\nПримеры органических строк БЕЗ типа:")
        examples = df_funnel[organic_mask & df_funnel['ТипАктивности'].isna()].head(5)
        print(examples[['Дата', 'Артикул', spend_col, shows_col, 'ТипАктивности']])

# ===== 6. Рекомендация =====
print("\n" + "=" * 80)
print("6. РЕКОМЕНДАЦИЯ")
print("=" * 80)

if 'Ozon ID' in df_voronka.columns:
    ozon_filled_pct = 100 * df_voronka['Ozon ID'].notna().sum() / len(df_voronka)
    
    if ozon_filled_pct > 80:
        print("\n💡 ВЫВОД: В воronka УЖЕ ЕСТЬ Ozon ID с хорошей заполненностью!")
        print("   Возможно, маппинг через df_reference не нужен.")
        print("   Попробуйте использовать Ozon ID напрямую.")
    else:
        print(f"\n⚠️ ВЫВОД: Ozon ID в воронке заполнен только на {ozon_filled_pct:.1f}%")
        print("   Нужен маппинг, НО возможно проблема в качестве данных.")
        
print("\n🔍 Основная проблема скорее всего в том, что:")
print("   1. Большая часть данных воронки - органика (без затрат)")
print("   2. Для органики ТипАктивности должен заполняться ПОСЛЕ мержа")
print("   3. Логика заполнения органики возможно не срабатывает")

print("\n" + "=" * 80)

УГЛУБЛЕННАЯ ДИАГНОСТИКА ПРОБЛЕМЫ

1. ИСХОДНАЯ ВОРОНКА (df_voronka)
--------------------------------------------------------------------------------
Размер: 3,075,806 строк
Колонки: ['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара', 'Показы в поиске и каталоге', 'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров', 'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму', 'В корзину из карточки товара', 'В корзину из поиска или каталога', 'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Ozon ID']

✅ В воронке УЖЕ ЕСТЬ колонка 'Ozon ID'!
  - Заполнено: 3,075,806 строк (100.0%)
  - Пусто (NaN): 0 строк

Примеры Ozon ID из воронки:
[149393764, 149390921, 149354561, 149372421, 149393771, 149393825, 149390977, 149390982, 149393806, 149390964]

Примеры Артикул из воронки:
['00006020', '00006080', '000060H0', '000060K0', '00006110', '00006130', '00006140', '00006170', '00006220', '00006250']

2. ЗАТРАТЫ (df_zatraty_enriched)
---------

In [51]:
df_funnel['ТипАктивности'].value_counts(dropna=False)

ТипАктивности
Органика       2660295
Трафареты       412036
NaN               3434
Вывод в топ         41
Name: count, dtype: int64

In [52]:
df_funnel[(df_funnel['ТипАктивности'].isna())&(df_funnel['Расход, ₽'] > 0.0)]

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Товары,Модель,Артикул OZ,"Расход, ₽",Рекламные показы,Рекламные показы на карточке товара,Рекламные заказано товаров,Рекламные заказано на сумму,ТипАктивности,Цена


In [53]:
df_union_reference = pd.merge(df_reference[['Артикул', 'Артикул OZ']], df_all_union, on="Артикул OZ", how='right')
# df_union_reference[df_union_reference['Артикул'].isna()]
df_union_reference

,Артикул,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт",Дата
0,NaN,3200383247,3200383267,3083.0,1,2026-02-28
1,NaN,1400496412,1400498686,1253.0,1,2026-02-28
2,NaN,3291358100,3291357496,3064.0,1,2026-02-28
3,NaN,1624110419,309743944,3094.0,1,2026-02-28
4,NaN,2402306650,2402306669,2819.0,1,2026-02-28
...,...,...,...,...,...,...
594724,NaN,1391271785,1917060944,2252.0,1,2026-03-30
594725,NaN,1628586928,1628618379,3140.0,1,2026-03-30
594726,NaN,2317085258,2317085209,2512.0,1,2026-03-30
594727,D5250710,3229631296,3229630844,1926.0,1,2026-03-30


In [54]:
df_all

,Артикул OZ,ID кампании,Инструмент,Место размещения,Дата
0,1857636566,18836810,Оплата за клик,Поиск и рекомендации,2026-02-28
1,1884357021,18842267,Оплата за клик,Поиск и рекомендации,2026-02-28
2,1400496303,18837211,Оплата за клик,Поиск и рекомендации,2026-02-28
3,1645388294,18838672,Оплата за клик,Поиск и рекомендации,2026-02-28
4,1074265121,18837920,Оплата за клик,Поиск и рекомендации,2026-02-28
...,...,...,...,...,...
487872,1627854891,18842748,Оплата за клик,NaN,2026-03-30
487873,2425022706,18834416,Оплата за клик,NaN,2026-03-30
487874,3632622106,23719604,Оплата за клик,NaN,2026-03-30
487875,1400495634,18844767,Оплата за клик,NaN,2026-03-30


In [55]:
df_union_reference['Продажи, ₽'] = pd.to_numeric(df_union_reference['Продажи, ₽'], errors='coerce')
df_union_reference['Заказы, шт'] = pd.to_numeric(df_union_reference['Заказы, шт'], errors='coerce')

df_union_agg = (
    df_union_reference
    .groupby(['Артикул', 'Дата'], as_index=False)
    .agg({
        'Артикул OZ': 'first',   # любой один из группы
        'SKU из объединенной карточки': 'first',   # любой один из группы
        'Продажи, ₽': 'sum',     # суммируем деньги
        'Заказы, шт': 'sum',     # суммируем заказы
    })
)
df_union_agg

,Артикул,Дата,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт"
0,01607050,2026-02-28,155173495,155173497,1534.0,1
1,01607050,2026-03-07,155173495,155173502,1534.0,1
2,01607050,2026-03-14,155173495,155173498,1534.0,1
3,064003S0,2026-04-01,3701585539,3701582518,2644.0,1
4,064092U0,2026-02-28,2430479050,2576284768,3511.0,2
...,...,...,...,...,...,...
68767,y8105200,2026-04-08,1383459568,1383459821,380.0,1
68768,y8105230,2026-03-01,1383459533,1383459821,818.0,2
68769,y8105230,2026-03-02,1383459533,1383459821,380.0,1
68770,y8105230,2026-03-03,1383459533,1383459568,1570.0,3


In [56]:
df_union_reference.columns

Index(['Артикул', 'Артикул OZ', 'SKU из объединенной карточки', 'Продажи, ₽',
       'Заказы, шт', 'Дата'],
      dtype='object')

In [57]:
df_union_reference.drop_duplicates(subset=['Артикул','Дата'])

,Артикул,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт",Дата
0,NaN,3200383247,3200383267,3083.0,1,2026-02-28
11,W7635151,1024902969,1024899930,7862.0,2,2026-02-28
12,W5259373,2330299694,2330300200,2243.0,1,2026-02-28
18,M6256154,1428832038,1428831647,3746.0,1,2026-02-28
19,W0650875,3190868183,3190868048,2364.0,1,2026-02-28
...,...,...,...,...,...,...
594700,W2160634,3291357240,3291358579,2179.0,1,2026-03-30
594706,S7856326,1400495634,1846577535,1121.0,1,2026-03-30
594714,W7107504,1627855684,1627856728,3900.0,1,2026-03-30
594716,W5204154,881692893,881691617,4392.0,1,2026-03-30


In [58]:
df_all=df_all.drop_duplicates(subset=['Дата','Артикул OZ'])

In [59]:
df_funnel.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Артикул OZ',
       'Расход, ₽', 'Рекламные показы', 'Рекламные показы на карточке товара',
       'Рекламные заказано товаров', 'Рекламные заказано на сумму',
       'ТипАктивности', 'Цена'],
      dtype='object')

In [60]:
import numpy as np
# --- утилита: выбрать колонку расхода (на всякий случай) ---
SPEND_CANDIDATES = ['Расход, ₽', 'Расход, руб', 'Расход, Р', 'Расход']
def _pick_spend_col(df: pd.DataFrame) -> str:
    for c in SPEND_CANDIDATES:
        if c in df.columns:
            return c
    raise KeyError(f"Не найдена колонка расхода среди: {SPEND_CANDIDATES}")
# 10. Связать "Воронка" с "Справочник" БЕЗ дублирования
try:
    print("Начинаем создавать таблицу ВоронкаСправочник...")
    start_time = time.time()

    # --- 0) Валидация входа ---
    required_columns = ["Дата", "Артикул"]
    for col in required_columns:
        if col not in df_funnel.columns:
            raise ValueError(f"Отсутствует столбец '{col}' в df_funnel.")

    # --- 1) Нормализуем ключ (точно так же в обеих таблицах) ---
    df_funnel = df_funnel.copy()
    df_reference = df_reference.copy()

    df_funnel["Артикул"] = (df_funnel["Артикул"].fillna('')
                                              .astype(str).str.strip().str.upper()
                                              .str[:8])
    df_reference["Артикул"] = (df_reference["Артикул"].fillna('')
                                                    .astype(str).str.strip().str.upper()
                                                    .str[:8])

    # Приведём дату в единый формат (без времени)
    # df_funnel["Дата"] = pd.to_datetime(df_funnel["Дата"], errors="coerce").dt.normalize()

    # --- 2) Оставляем только нужные поля справочника ---
    reference_columns = [
        "Артикул", "Артикул OZ", "Наименование", "Коллекция", "Бренд", "Сезон", "Направление",
        "Розничный отдел", "Модель", "Группа", "Бизнес-группа", "Техсегмент",
        "Байер", "Две последние коллекции", "Основной артикул", "Себестоимость с НДС",
        "Процент выкупа", "НДС", "Ответственный за группу", "Группа для отчетов"
    ]
    # оставим только реально существующие колонки
    reference_columns = [c for c in reference_columns if c in df_reference.columns]
    df_reference_filtered = df_reference[["Артикул"] + [c for c in reference_columns if c != "Артикул"]].copy()

    # --- 3) Делаем справочник УНИКАЛЬНЫМ по Артикулу (one-row-per-Артикул) ---
    # Если есть дубликаты одного артикула — берём первую строку (можно заменить на приоритетное правило)
    dups = df_reference_filtered["Артикул"].duplicated(keep=False).sum()
    if dups:
        print(f"[INFO] В справочнике обнаружены дубликаты по 'Артикул' (после .str[:8]): {dups} строк.")
    ref_unique = (df_reference_filtered
                  .sort_values(["Артикул"])
                  .drop_duplicates(subset=["Артикул"], keep="first")
                  .reset_index(drop=True))

    # sanity: строго уникально
    assert not ref_unique["Артикул"].duplicated().any(), "ref_unique всё ещё содержит дубликаты Артикул"

    # --- 4) Контроль инвариантов расхода ДО merge ---
    try:
        spend_col = _pick_spend_col(df_funnel)
    except KeyError:
        spend_col = None

    if spend_col:
        before_total = pd.to_numeric(df_funnel[spend_col], errors='coerce').sum()
        before_by_date = (df_funnel.groupby("Дата", as_index=False)[spend_col]
                                   .sum(min_count=1)
                                   .rename(columns={spend_col: "Расход_до"}))
    else:
        print("[INFO] В df_funnel нет колонки расхода — инварианты по расходу не проверяем.")

    del ref_unique['Артикул OZ']
    del ref_unique['Модель']
    # --- 5) LEFT-merge строго m:1 (исключает размножение строк) ---
    df_funnel_reference = pd.merge(
        df_funnel,
        ref_unique,
        on="Артикул",
        how="left",
        validate="m:1",     # если справа снова появятся дубликаты — упадёт сразу
        indicator=False
    )

    # Ничего НЕ дропаем из df_funnel_reference! (drop_duplicates ломает суммы)

    # --- 6) Контроль ПОСЛЕ merge (сумма не должна измениться ни по датам, ни итого) ---
    if spend_col:
        after_total = pd.to_numeric(df_funnel_reference[spend_col], errors='coerce').sum()
        after_by_date = (df_funnel_reference.groupby("Дата", as_index=False)[spend_col]
                                          .sum(min_count=1)
                                          .rename(columns={spend_col: "Расход_после"}))
        check = before_by_date.merge(after_by_date, on="Дата", how="outer").fillna(0)
        drift = check.loc[~np.isclose(check["Расход_до"], check["Расход_после"], rtol=1e-9, atol=1e-6)]

        print(f"[CHECK] Общая сумма расхода: до={before_total:,.2f} | после={after_total:,.2f}")
        if len(drift):
            print("[WARN] Обнаружены расхождения по датам (первые 10):")
            print(drift.head(10))
            # Если тут расхождения — значит есть иная проблема (например, обрезка .str[:8] склеила разные артикула)
    else:
        after_total = None

    # --- 7) Формат вывода даты (если нужен именно date без времени) ---
    # df_funnel_reference["Дата"] = pd.to_datetime(df_funnel_reference["Дата"]).dt.date

    # Итог
    print("Первые 5 строк таблицы ВоронкаСправочник:")
    print(df_funnel_reference.head())

    elapsed_time = time.time() - start_time
    print(f"Таблица ВоронкаСправочник успешно создана. Время выполнения: {elapsed_time:.2f} c.")

except Exception as e:
    print(f"Ошибка при создании таблицы ВоронкаСправочник: {e}")

Начинаем создавать таблицу ВоронкаСправочник...
[INFO] В справочнике обнаружены дубликаты по 'Артикул' (после .str[:8]): 114 строк.
[CHECK] Общая сумма расхода: до=274,030,981.94 | после=274,030,981.94
Первые 5 строк таблицы ВоронкаСправочник:
        Дата   Артикул  Показы, всего  Показы на карточке товара  \
0 2026-02-28  00006020              2                          0   
1 2026-02-28  00006080              2                          0   
2 2026-02-28  000060H0              2                          0   
3 2026-02-28  000060K0              2                          0   
4 2026-02-28  00006110              2                          0   

   Показы в поиске и каталоге  Позиция в поиске и каталоге  В корзину, всего  \
0                           0                          0.0                 0   
1                           0                          0.0                 0   
2                           0                          0.0                 0   
3                          

In [61]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Артикул OZ',
       'Расход, ₽', 'Рекламные показы', 'Рекламные показы на карточке товара',
       'Рекламные заказано товаров', 'Рекламные заказано на сумму',
       'ТипАктивности', 'Цена', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
       'Направление', 'Розничный отдел', 'Группа', 'Бизнес-группа',
       'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов'],
      dtype='object')

In [62]:
df_funnel_reference['ТипАктивности'].value_counts()

ТипАктивности
Органика       2660295
Трафареты       412036
Вывод в топ         41
Name: count, dtype: int64

In [63]:
df_funnel[df_funnel['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.int64(0)

In [64]:
df_funnel[df_funnel['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(0.0)

In [65]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(0.0)

In [66]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.int64(0)

In [67]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2026-02-28,00006020,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006020,388.6815,0.935463,20.0,Прусс Константин,Обувь
1,2026-02-28,00006080,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006080,415.6297,0.935463,20.0,Прусс Константин,Обувь
2,2026-02-28,000060H0,2,0,0,0.00,0,0,1,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060H0,1034.0710,0.935463,20.0,Селютина Арина,Обувь
3,2026-02-28,000060K0,2,0,0,0.00,0,0,0,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060K0,1034.0710,0.935463,20.0,Селютина Арина,Обувь
4,2026-02-28,00006110,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006110,338.2640,0.935463,20.0,Прусс Константин,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3075801,2026-04-27,Y9808110,220,24,67,892.37,3,0,0,0,...,Одежда и аксессуары,NaN,Коновалова И. Одежда,2025SS,y9808110,822.0091,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075802,2026-04-27,Y9808120,256,25,59,1153.39,1,0,0,0,...,Одежда и аксессуары,NaN,Коновалова И. Одежда,2025SS,y9808120,846.4499,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075803,2026-04-27,Y9808130,124,11,35,524.67,0,0,0,0,...,Одежда и аксессуары,NaN,NaN,2025SS,y9808130,887.7361,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075804,2026-04-27,Y9808140,169,19,56,443.09,2,1,0,0,...,Одежда и аксессуары,NaN,NaN,2025SS,y9808140,885.4398,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда


In [68]:
df_funnel_reference[df_funnel_reference['Дата'] == "2025-12-28"]

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов


In [69]:
df_funnel_reference['Расход, ₽'] = pd.to_numeric(
    df_funnel_reference['Расход, ₽']
        .astype(str)
        .str.replace('\u00A0', '', regex=False)  # NBSP
        .str.replace(' ',      '', regex=False)  # обычные пробелы
        .str.replace('₽',      '', regex=False)
        .str.replace(',',      '.', regex=False),  # ВАЖНО: str.replace, не replace
    errors='coerce'
)

In [70]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2026-02-28,00006020,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006020,388.6815,0.935463,20.0,Прусс Константин,Обувь
1,2026-02-28,00006080,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006080,415.6297,0.935463,20.0,Прусс Константин,Обувь
2,2026-02-28,000060H0,2,0,0,0.00,0,0,1,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060H0,1034.0710,0.935463,20.0,Селютина Арина,Обувь
3,2026-02-28,000060K0,2,0,0,0.00,0,0,0,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060K0,1034.0710,0.935463,20.0,Селютина Арина,Обувь
4,2026-02-28,00006110,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006110,338.2640,0.935463,20.0,Прусс Константин,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3075801,2026-04-27,Y9808110,220,24,67,892.37,3,0,0,0,...,Одежда и аксессуары,NaN,Коновалова И. Одежда,2025SS,y9808110,822.0091,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075802,2026-04-27,Y9808120,256,25,59,1153.39,1,0,0,0,...,Одежда и аксессуары,NaN,Коновалова И. Одежда,2025SS,y9808120,846.4499,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075803,2026-04-27,Y9808130,124,11,35,524.67,0,0,0,0,...,Одежда и аксессуары,NaN,NaN,2025SS,y9808130,887.7361,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075804,2026-04-27,Y9808140,169,19,56,443.09,2,1,0,0,...,Одежда и аксессуары,NaN,NaN,2025SS,y9808140,885.4398,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда


In [71]:
df_union_agg#.drop_duplicates(subset=['Артикул OZ', 'SKU из объединенной карточки'])

,Артикул,Дата,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт"
0,01607050,2026-02-28,155173495,155173497,1534.0,1
1,01607050,2026-03-07,155173495,155173502,1534.0,1
2,01607050,2026-03-14,155173495,155173498,1534.0,1
3,064003S0,2026-04-01,3701585539,3701582518,2644.0,1
4,064092U0,2026-02-28,2430479050,2576284768,3511.0,2
...,...,...,...,...,...,...
68767,y8105200,2026-04-08,1383459568,1383459821,380.0,1
68768,y8105230,2026-03-01,1383459533,1383459821,818.0,2
68769,y8105230,2026-03-02,1383459533,1383459821,380.0,1
68770,y8105230,2026-03-03,1383459533,1383459568,1570.0,3


In [72]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Артикул OZ',
       'Расход, ₽', 'Рекламные показы', 'Рекламные показы на карточке товара',
       'Рекламные заказано товаров', 'Рекламные заказано на сумму',
       'ТипАктивности', 'Цена', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
       'Направление', 'Розничный отдел', 'Группа', 'Бизнес-группа',
       'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов'],
      dtype='object')

In [73]:
# df_funnel_reference[df_funnel_reference[["Дата", "Артикул OZ"]].isin(df_all[["Дата", "Артикул OZ"]])]#.dropna(subset="Дата")

In [74]:
df_all

,Артикул OZ,ID кампании,Инструмент,Место размещения,Дата
0,1857636566,18836810,Оплата за клик,Поиск и рекомендации,2026-02-28
1,1884357021,18842267,Оплата за клик,Поиск и рекомендации,2026-02-28
2,1400496303,18837211,Оплата за клик,Поиск и рекомендации,2026-02-28
3,1645388294,18838672,Оплата за клик,Поиск и рекомендации,2026-02-28
4,1074265121,18837920,Оплата за клик,Поиск и рекомендации,2026-02-28
...,...,...,...,...,...
487872,1627854891,18842748,Оплата за клик,NaN,2026-03-30
487873,2425022706,18834416,Оплата за клик,NaN,2026-03-30
487874,3632622106,23719604,Оплата за клик,NaN,2026-03-30
487875,1400495634,18844767,Оплата за клик,NaN,2026-03-30


In [75]:
df_union_agg

,Артикул,Дата,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт"
0,01607050,2026-02-28,155173495,155173497,1534.0,1
1,01607050,2026-03-07,155173495,155173502,1534.0,1
2,01607050,2026-03-14,155173495,155173498,1534.0,1
3,064003S0,2026-04-01,3701585539,3701582518,2644.0,1
4,064092U0,2026-02-28,2430479050,2576284768,3511.0,2
...,...,...,...,...,...,...
68767,y8105200,2026-04-08,1383459568,1383459821,380.0,1
68768,y8105230,2026-03-01,1383459533,1383459821,818.0,2
68769,y8105230,2026-03-02,1383459533,1383459821,380.0,1
68770,y8105230,2026-03-03,1383459533,1383459568,1570.0,3


In [76]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2026-02-28,00006020,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006020,388.6815,0.935463,20.0,Прусс Константин,Обувь
1,2026-02-28,00006080,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006080,415.6297,0.935463,20.0,Прусс Константин,Обувь
2,2026-02-28,000060H0,2,0,0,0.00,0,0,1,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060H0,1034.0710,0.935463,20.0,Селютина Арина,Обувь
3,2026-02-28,000060K0,2,0,0,0.00,0,0,0,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060K0,1034.0710,0.935463,20.0,Селютина Арина,Обувь
4,2026-02-28,00006110,2,0,0,0.00,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006110,338.2640,0.935463,20.0,Прусс Константин,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3075801,2026-04-27,Y9808110,220,24,67,892.37,3,0,0,0,...,Одежда и аксессуары,NaN,Коновалова И. Одежда,2025SS,y9808110,822.0091,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075802,2026-04-27,Y9808120,256,25,59,1153.39,1,0,0,0,...,Одежда и аксессуары,NaN,Коновалова И. Одежда,2025SS,y9808120,846.4499,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075803,2026-04-27,Y9808130,124,11,35,524.67,0,0,0,0,...,Одежда и аксессуары,NaN,NaN,2025SS,y9808130,887.7361,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда
3075804,2026-04-27,Y9808140,169,19,56,443.09,2,1,0,0,...,Одежда и аксессуары,NaN,NaN,2025SS,y9808140,885.4398,1.000000,20.0,Зиннуров Ильнур,Взрослая одежда


In [77]:
# df_all['Дата'] = pd.to_datetime(df_all['Дата'])
# # объединяем с df_funnel
# df_result = df_funnel_reference.merge(
#     df_all.rename(columns=),
#     on=["Дата", "Артикул OZ"],
#     how="left"
# )

df_union_agg['Дата'] = pd.to_datetime(df_union_agg['Дата'])
df_funnel_reference = df_funnel_reference.merge(
    df_union_agg,
    on=["Дата", "Артикул OZ"],
    how="left"
)
# df_result['ТипАктивности'] = pd.Series()

# # Трафарет
# mask = (
#     ((df_result['Инструмент'] == 'Оплата за клик') &
#     (df_result['Место размещения'] == 'Поиск и рекомендации')) #|
#     #(df_result['ТипАктивности'] == 'Оплата за клик')
# )
# df_result.loc[mask, 'ТипАктивности'] = 'Трафареты'

# # Вывод в топ
# mask = (
#     ((df_result['Инструмент'] == 'Оплата за клик') & 
#      (df_result['Место размещения'] == 'Поиск')) #|
#     #(df_result['ТипАктивности'] == 'ТОП')
# )
# df_result.loc[mask, 'ТипАктивности'] = 'Вывод в топ'

# # Органика
# mask = (
#     ((df_result['Расход, ₽'] == 0) | 
#      (df_result['Расход, ₽'] == 0.0))
# )
# df_result.loc[mask, 'ТипАктивности'] = 'Органика'

In [78]:
df_funnel_reference.drop(columns=['Артикул_y'], inplace=True)
df_funnel_reference.rename(columns={'Артикул_x':'Артикул'}, inplace=True)
# df_funnel_reference.drop(columns=['Модель_y'], inplace=True)
# df_funnel_reference.rename(columns={'Модель_x':'Модель'}, inplace=True)

In [79]:
# df_result

In [80]:
df_funnel_reference.rename(columns={'Продажи, ₽':'Ассоциированные заказы, руб', 'Заказы, шт':'Ассоциированные заказы, шт'}, inplace=True)

In [81]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(0.0)

In [82]:
funnel_columns = ['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'
       ]
funnel_columns_all = [
    'Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
    'Показы на карточке товара', 'Показы в поиске и каталоге',
    'Позиция в поиске и каталоге', 'В корзину, всего',
    'Заказано товаров', 'Отменено товаров', 'Доставлено товаров',
    'Возвращено товаров', 'Заказано на сумму',
    'В корзину из карточки товара', 'Выкупили ШТ',
    'Расход, ₽', 'Рекламные заказано на сумму',
    'Рекламные заказано товаров', 'Рекламные показы',
    'Рекламные показы на карточке товара', 'Цена',
    'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
    'Направление', 'Розничный отдел', 'Модель', 'Группа',
    'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции',
    'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС',
    'Ответственный за группу', 'Группа для отчетов',
    'ID кампании', 'Инструмент', 'Место размещения'
]
funnel_columns_widing = ['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'
       ]

In [83]:
import numpy as np
import pandas as pd

def build_funnel_wide(
    df_raw: pd.DataFrame,
    funnel_columns: list,
    all_types=('Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика'),
    infer_organic_by_zero_spend=False,
    spend_col='Расход, ₽',
    extra_agg='first',          # 'first' | 'join'
    extra_join_sep=' | ',
    check_spend_invariance=True,
    atol=1e-6, rtol=1e-9
):
    """
    Склеивает строки по (Дата, Артикул), раскладывает метрики по типам активности,
    добавляет ИТОГИ, которые считаются напрямую из исходника по ключу (Дата, Артикул).
    Благодаря этому 'Расход, ₽' (и прочие итоги) сохраняют исходные значения.
    """

    # ---- 0) Исходная выборка для расчётов (только нужные колонки) ----
    cols_present = [c for c in funnel_columns if c in df_raw.columns]
    df = df_raw.loc[:, cols_present].copy()

    # ---- 1) Нормализуем тип активности ----
    type_col = 'ТипАктивности'
    df[type_col] = df[type_col].replace({'ТОП': 'Вывод в топ'})
    if infer_organic_by_zero_spend and spend_col in df.columns:
        m0 = pd.to_numeric(df[spend_col], errors='coerce').fillna(0).eq(0)
        df.loc[m0, type_col] = 'Органика'

    # ---- 2) Ключи/метрики и приведение типов ----
    key_cols = ['Дата', 'Артикул']
    met_start = funnel_columns.index(type_col) + 1
    metric_cols = [c for c in funnel_columns[met_start:] if c in df.columns]

    # аккуратно приводим метрики
    for c in metric_cols:
        if c == spend_col:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')   # спенд в float64
        else:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    # ---- 3) ИТОГИ БЕЗ ПРЕФИКСОВ (ИСТИНА) по (Дата, Артикул) ----
    totals_df = (df.groupby(key_cols, as_index=False)[metric_cols]
                   .sum(min_count=1))   # если где-то все NaN, останется NaN; это корректно

    # ---- 4) Префиксные метрики по типам ----
    g = (df.groupby(key_cols + [type_col], as_index=False)[metric_cols]
           .sum(min_count=1))

    # Дополним all_types тем, что реально встретилось
    types_present = g[type_col].dropna().unique().tolist()
    all_types = list(dict.fromkeys(list(all_types) + [t for t in types_present if t not in all_types]))

    # Полная база ключей = все пары (Дата, Артикул), которые встречаются в исходнике
    base = (df[key_cols].drop_duplicates()
                    .set_index(key_cols)
                    .sort_index())

    # Сформируем блоки префиксных метрик и флаги наличия типов
    metric_blocks, flag_blocks = [], []
    for t in all_types:
        sub = g[g[type_col] == t].set_index(key_cols)

        if sub.empty:
            # пустой тип → нули на всю базу
            sub_metrics = pd.DataFrame(
                0.0, index=base.index,
                columns=[f'{t}_{m}' for m in metric_cols],
                dtype='float32'
            )
        else:
            sub_metrics = (sub[metric_cols]
                           .rename(columns={m: f'{t}_{m}' for m in metric_cols})
                           .reindex(base.index, fill_value=0.0))

            # типы данных: спенд оставляем float64
            for col in sub_metrics.columns:
                if col.endswith(spend_col):
                    sub_metrics[col] = sub_metrics[col].astype('float64')
                else:
                    sub_metrics[col] = sub_metrics[col].astype('float32')

        metric_blocks.append(sub_metrics)

        # бинарный флаг присутствия типа (на уровне ключа)
        flag = pd.Series(1, index=sub.index, name=t) if not sub.empty else pd.Series(0, index=base.index, name=t)
        flag_blocks.append(flag.reindex(base.index, fill_value=0).astype('int8'))

    metrics_block = pd.concat(metric_blocks, axis=1)
    flags_block   = pd.concat(flag_blocks, axis=1)

    # ---- 5) СБОРКА CORE: ключи + флаги + префиксные метрики + ИТОГИ ИЗ totals_df ----
    core = pd.concat(
        [
            base.reset_index(),
            flags_block.reset_index(drop=True),
            metrics_block.reset_index(drop=True)
        ],
        axis=1
    )

    # присоединяем ИТОГИ (истина) строго m:1
    core = core.merge(totals_df, on=key_cols, how='left', validate='m:1')
    # === НОВЫЙ КОД: Сохранение доминирующего типа ===
    # Для каждого ключа выбираем тип с максимальным расходом
    # if spend_col in g.columns:
    #     dominant_type = (
    #         g.sort_values([*key_cols, spend_col], ascending=[True, True, False])
    #         .groupby(key_cols, as_index=False)
    #         .first()[[*key_cols, type_col]]
    #     )
    # else:
    #     # Если расходов нет - берем первый встретившийся тип
    #     dominant_type = (
    #         g.groupby(key_cols, as_index=False)
    #         .first()[[*key_cols, type_col]]
    #     )
    # core = core.merge(dominant_type, on=key_cols, how='left', validate='m:1')
    
    # ---- 6) ДОП. колонки из df_raw (НЕ участвуют в расчётах) ----
    exclude = set(key_cols + metric_cols)
    extra_cols = [c for c in df_raw.columns if c not in exclude]
    if extra_cols:
        if extra_agg == 'first':
            dims_block = (df_raw[key_cols + extra_cols]
                            .sort_values(key_cols)
                            .groupby(key_cols, as_index=False)
                            .first())
        elif extra_agg == 'join':
            def _join_unique(s):
                v = pd.unique(s.dropna().astype(str))
                return extra_join_sep.join(v) if len(v) else np.nan
            dims_block = (df_raw[key_cols + extra_cols]
                            .groupby(key_cols, as_index=False)
                            .agg({c: _join_unique for c in extra_cols}))
        else:
            raise ValueError("extra_agg должен быть 'first' или 'join'")

        out = core.merge(dims_block, on=key_cols, how='left', validate='m:1')
    else:
        out = core

    # ---- 7) Проверка инварианта для 'Расход, ₽' (опционально) ----
    if check_spend_invariance and (spend_col in totals_df.columns):
        base_sp = (totals_df.groupby(key_cols, as_index=False)[spend_col].sum(min_count=1)
                             .rename(columns={spend_col: '__base__'}))
        after_sp = (out.groupby(key_cols, as_index=False)[spend_col].sum(min_count=1)
                          .rename(columns={spend_col: '__after__'}))
        chk = base_sp.merge(after_sp, on=key_cols, how='outer').fillna(0)
        bad = chk.loc[~np.isclose(chk['__base__'], chk['__after__'], rtol=rtol, atol=atol)]
        if not bad.empty:
            print("[WARN] Инвариант по 'Расход, ₽' нарушен для некоторых ключей (первые 10):")
            print(bad.head(10))

    # ---- 8) Порядок колонок: ключи → доп.колонки → флаги → префиксные метрики → ИТОГИ ----
    ordered = []
    ordered += key_cols
    ordered += type_col
    ordered += [c for c in df_raw.columns if (c in out.columns and c not in key_cols and c not in (metric_cols))]
    ordered += [t for t in all_types if t in out.columns]

    for m in metric_cols:
        # префиксные
        ordered += [f'{t}_{m}' for t in all_types if f'{t}_{m}' in out.columns]
    # ИТОГИ (без префикса) — в самом конце блоком в исходном порядке
    ordered += [m for m in metric_cols if m in out.columns]

    out = out[[c for c in ordered if c in out.columns]].copy()
    return out

In [84]:
final_after_widing_columns = ['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталоге', 'Трафарет_Позиция в поиске и каталоге', 'Оплата за заказ_Позиция в поиске и каталоге', 'Органика_Позиция в поиске и каталоге', 'Вывод в топ_В корзину, всего', 'Трафарет_В корзину, всего', 'Оплата за заказ_В корзину, всего', 'Органика_В корзину, всего', 'Вывод в топ_Заказано товаров', 'Трафарет_Заказано товаров', 'Оплата за заказ_Заказано товаров', 'Органика_Заказано товаров', 'Вывод в топ_Отменено товаров', 'Трафарет_Отменено товаров', 'Оплата за заказ_Отменено товаров', 'Органика_Отменено товаров', 'Вывод в топ_Доставлено товаров', 'Трафарет_Доставлено товаров', 'Оплата за заказ_Доставлено товаров', 'Органика_Доставлено товаров', 'Вывод в топ_Возвращено товаров', 'Трафарет_Возвращено товаров', 'Оплата за заказ_Возвращено товаров', 'Органика_Возвращено товаров', 'Вывод в топ_Заказано на сумму', 'Трафарет_Заказано на сумму', 'Оплата за заказ_Заказано на сумму', 'Органика_Заказано на сумму', 'Вывод в топ_В корзину из карточки товара', 'Трафарет_В корзину из карточки товара', 'Оплата за заказ_В корзину из карточки товара', 'Органика_В корзину из карточки товара', 'Вывод в топ_Выкупили ШТ', 'Трафарет_Выкупили ШТ', 'Оплата за заказ_Выкупили ШТ', 'Органика_Выкупили ШТ', 'Вывод в топ_Расход, ₽', 'Трафарет_Расход, ₽', 'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽', 'Вывод в топ_Рекламные заказано на сумму', 'Трафарет_Рекламные заказано на сумму', 'Оплата за заказ_Рекламные заказано на сумму', 'Органика_Рекламные заказано на сумму', 'Вывод в топ_Рекламные заказано товаров', 'Трафарет_Рекламные заказано товаров', 'Оплата за заказ_Рекламные заказано товаров', 'Органика_Рекламные заказано товаров', 'Вывод в топ_Рекламные показы', 'Трафарет_Рекламные показы', 'Оплата за заказ_Рекламные показы', 'Органика_Рекламные показы', 'Вывод в топ_Рекламные показы на карточке товара', 'Трафарет_Рекламные показы на карточке товара', 'Оплата за заказ_Рекламные показы на карточке товара', 'Органика_Рекламные показы на карточке товара', 'Вывод в топ_Цена', 'Трафарет_Цена', 'Оплата за заказ_Цена', 'Органика_Цена', 'Показы, всего', 'Показы на карточке товара', 'Показы в поиске и каталоге', 'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров', 'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽', 'Рекламные заказано на сумму', 'Рекламные заказано товаров', 'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена']

In [85]:
out = build_funnel_wide(df_raw=df_funnel_reference, funnel_columns=funnel_columns_widing)
out = out[final_after_widing_columns]
out

,Дата,Артикул,Артикул OZ,ТипАктивности,Наименование,Коллекция,Бренд,Сезон,Направление,Розничный отдел,...,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2026-02-28,00006020,149393764,Органика,Балетки женские 298-9CK,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,569.0
1,2026-02-28,00006080,149390921,Органика,Балетки женские ZS189-7K,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,690.0
2,2026-02-28,000060H0,149354561,Органика,Балетки женские K0346PM-1A,2019SS,PIERRE CARDIN,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,-1.0,0.0,NaN,NaN,NaN,NaN,846.0
3,2026-02-28,000060K0,149372421,Органика,Балетки женские K0346PM-1E,2019SS,PIERRE CARDIN,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,846.0
4,2026-02-28,00006110,149393771,Органика,Балетки женские 299-8K,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,495.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3075800,2026-04-27,Y9808110,1997932756,Органика,Шорты мужские A85512-2,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,...,0.0,0.0,3.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3075801,2026-04-27,Y9808120,1997933206,Органика,Шорты мужские A85512-3,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,...,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3075802,2026-04-27,Y9808130,1934644792,Органика,Шорты мужские SS25C2020,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3075803,2026-04-27,Y9808140,1950209153,Органика,Шорты мужские SS25C2021,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",...,0.0,1768.0,2.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN


In [86]:
print(len(list(out.columns)))

119


In [87]:
print(list(out.columns))

['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталоге', 'Трафарет_Позиция в поиске и

In [88]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(0.0)

In [89]:
out[out['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(0.0)

In [90]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.int64(0)

In [91]:
out[out['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.float32(0.0)

In [92]:
df_funnel_reference = out

In [93]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Наименование',
       'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел',
       ...
       'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽',
       'Рекламные заказано на сумму', 'Рекламные заказано товаров',
       'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена'],
      dtype='object', length=119)

In [94]:
df_funnel_reference

,Дата,Артикул,Артикул OZ,ТипАктивности,Наименование,Коллекция,Бренд,Сезон,Направление,Розничный отдел,...,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2026-02-28,00006020,149393764,Органика,Балетки женские 298-9CK,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,569.0
1,2026-02-28,00006080,149390921,Органика,Балетки женские ZS189-7K,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,690.0
2,2026-02-28,000060H0,149354561,Органика,Балетки женские K0346PM-1A,2019SS,PIERRE CARDIN,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,-1.0,0.0,NaN,NaN,NaN,NaN,846.0
3,2026-02-28,000060K0,149372421,Органика,Балетки женские K0346PM-1E,2019SS,PIERRE CARDIN,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,846.0
4,2026-02-28,00006110,149393771,Органика,Балетки женские 299-8K,2019SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,495.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3075800,2026-04-27,Y9808110,1997932756,Органика,Шорты мужские A85512-2,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,...,0.0,0.0,3.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3075801,2026-04-27,Y9808120,1997933206,Органика,Шорты мужские A85512-3,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,...,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3075802,2026-04-27,Y9808130,1934644792,Органика,Шорты мужские SS25C2020,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3075803,2026-04-27,Y9808140,1950209153,Органика,Шорты мужские SS25C2021,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",...,0.0,1768.0,2.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN


In [95]:
# 11. Связать "ВоронкаСправочник" с "Остатки с дистрибуцией"
try:
    print("Начинаем создавать таблицу ДБбезПризнаков...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution['Дата'] = pd.to_datetime(df_stock_with_distribution['Дата'])
    df_final_db = pd.merge(df_funnel_reference, df_stock_with_distribution, left_on=["Дата", "Артикул"], right_on=["Дата", "Артикул"], how="left")
    df_final_db = format_date_column(df_final_db, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБбезПризнаков:")
    print(df_final_db.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБбезПризнаков успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБбезПризнаков: {e}")

Начинаем создавать таблицу ДБбезПризнаков...
Первые 5 строк таблицы ДБбезПризнаков:
         Дата   Артикул Артикул OZ ТипАктивности                Наименование  \
0  2026-02-28  00006020  149393764      Органика     Балетки женские 298-9CK   
1  2026-02-28  00006080  149390921      Органика    Балетки женские ZS189-7K   
2  2026-02-28  000060H0  149354561      Органика  Балетки женские K0346PM-1A   
3  2026-02-28  000060K0  149372421      Органика  Балетки женские K0346PM-1E   
4  2026-02-28  00006110  149393771      Органика      Балетки женские 299-8K   

  Коллекция          Бренд            Сезон    Направление Розничный отдел  \
0    2019SS     T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь   
1    2019SS     T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь   
2    2019SS  PIERRE CARDIN  лето (закрытое)  Женская обувь   Женская обувь   
3    2019SS  PIERRE CARDIN  лето (закрытое)  Женская обувь   Женская обувь   
4    2019SS     T.TACCARDI  лето (закрытое)  

In [96]:
df_final_db.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Наименование',
       'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел',
       ...
       'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽',
       'Рекламные заказано на сумму', 'Рекламные заказано товаров',
       'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена',
       'Остаток Агрегатора', 'Дистрибуция'],
      dtype='object', length=121)

In [97]:
# 5. Получить данные из файла !!!_Признаки для артикула и даты для Озон
try:
    print("Начинаем получать данные для Признаков...")
    start_time = time.time()  # Запускаем таймер
    file_path_features = os.path.join(FOLDER_PATH_FEATURES, "!!!_Признаки для артикула и даты для Озон.xlsx")
    if os.path.exists(file_path_features):
        df_item_features = pd.read_excel(file_path_features, sheet_name="Признаки для артикула", dtype=str, engine="calamine")
        df_date_features = pd.read_excel(file_path_features, sheet_name="Признаки для дат", dtype={0: "datetime64[ns]", **{i: str for i in range(1, 6)}}, engine="calamine")

        # Обработка ошибок
        df_item_features = handle_errors(df_item_features)
        df_date_features = handle_errors(df_date_features)

        # Форматирование даты
        df_date_features = format_date_column(df_date_features, 'Дата')

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки артикула:")
        print(df_item_features.head())

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки дат:")
        print(df_date_features.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Признаков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл '!!!_Признаки для артикула и даты для Озон.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Признаков: {e}")

Начинаем получать данные для Признаков...
Первые 5 строк таблицы Признаки артикула:
    Артикул Признак Артикула 1 Признак Артикула 2 Признак Артикула 3  \
0  00001851                NaN                NaN                NaN   
1  00001852                NaN                NaN                NaN   
2  00001855                NaN                NaN                NaN   
3  00001856                NaN                NaN                NaN   
4  00001931                NaN                NaN                NaN   

  Признак Артикула 4 Признак Артикула 5  
0                NaN                NaN  
1                NaN                NaN  
2                NaN                NaN  
3                NaN                NaN  
4                NaN                NaN  
Первые 5 строк таблицы Признаки дат:
Empty DataFrame
Columns: [Дата, Признак Даты 1, Признак Даты 2, Признак Даты 3, Признак Даты 4, Признак Даты 5]
Index: []
Данные для Признаков успешно сохранены. Время выполнения: 0 часа(ов) 0 м

In [98]:
# 12. Связать "ДБбезПризнаков" с "Признаки для артикула"
try:
    print("Начинаем создавать таблицу ДБсПризнакамиАртикула...")
    start_time = time.time()  # Запускаем таймер
    df_final_db_item_features = pd.merge(df_final_db, df_item_features, left_on=["Артикул"], right_on=["Артикул"], how="left")
    df_final_db_item_features = format_date_column(df_final_db_item_features, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПризнакамиАртикула:")
    print(df_final_db_item_features.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПризнакамиАртикула успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнакамиАртикула: {e}")

Начинаем создавать таблицу ДБсПризнакамиАртикула...
Первые 5 строк таблицы ДБсПризнакамиАртикула:
         Дата   Артикул Артикул OZ ТипАктивности                Наименование  \
0  2026-02-28  00006020  149393764      Органика     Балетки женские 298-9CK   
1  2026-02-28  00006080  149390921      Органика    Балетки женские ZS189-7K   
2  2026-02-28  000060H0  149354561      Органика  Балетки женские K0346PM-1A   
3  2026-02-28  000060K0  149372421      Органика  Балетки женские K0346PM-1E   
4  2026-02-28  00006110  149393771      Органика      Балетки женские 299-8K   

  Коллекция          Бренд            Сезон    Направление Розничный отдел  \
0    2019SS     T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь   
1    2019SS     T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь   
2    2019SS  PIERRE CARDIN  лето (закрытое)  Женская обувь   Женская обувь   
3    2019SS  PIERRE CARDIN  лето (закрытое)  Женская обувь   Женская обувь   
4    2019SS     T.TACCARDI  лет

In [99]:
# 13. Связать "ДБсПризнакамиАртикула" с "Признаки для дат"
try:
    print("Начинаем создавать таблицу ДБсПризнаками...")
    start_time = time.time()
    df_final_db_all_features = pd.merge(df_final_db_item_features, df_date_features, on="Дата", how="left")
    df_final_db_all_features = format_date_column(df_final_db_all_features, 'Дата')

    print("Первые 5 строк таблицы ДБсПризнаками:")
    print(df_final_db_all_features.head())

    # Сохранение финальной таблицы
    # df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_New.csv"), index=False)

    elapsed_time = time.time() - start_time
    print(f"Таблица ДБсПризнаками успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнаками: {e}")

Начинаем создавать таблицу ДБсПризнаками...
Первые 5 строк таблицы ДБсПризнаками:
         Дата   Артикул Артикул OZ ТипАктивности                Наименование  \
0  2026-02-28  00006020  149393764      Органика     Балетки женские 298-9CK   
1  2026-02-28  00006080  149390921      Органика    Балетки женские ZS189-7K   
2  2026-02-28  000060H0  149354561      Органика  Балетки женские K0346PM-1A   
3  2026-02-28  000060K0  149372421      Органика  Балетки женские K0346PM-1E   
4  2026-02-28  00006110  149393771      Органика      Балетки женские 299-8K   

  Коллекция          Бренд            Сезон    Направление Розничный отдел  \
0    2019SS     T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь   
1    2019SS     T.TACCARDI  лето (закрытое)  Женская обувь   Женская обувь   
2    2019SS  PIERRE CARDIN  лето (закрытое)  Женская обувь   Женская обувь   
3    2019SS  PIERRE CARDIN  лето (закрытое)  Женская обувь   Женская обувь   
4    2019SS     T.TACCARDI  лето (закрытое)  Же

In [100]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(0.0)

In [101]:
print(list(df_final_db_all_features.columns))

['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталоге', 'Трафарет_Позиция в поиске и

In [102]:
import numpy as np
# df — ваша широкая таблица с флагами типов (1/0)
type_order_paid = ['Вывод в топ', 'Трафарет', 'Оплата за заказ']
paid_cols = [c for c in type_order_paid if c in df_final_db_all_features.columns]  # на случай отсутствующих

# Матрица флагов платных типов
flags = df_final_db_all_features[paid_cols].fillna(0).astype('uint8').to_numpy()
labels = np.array(paid_cols, dtype=object)

# Собираем подписи для платных комбинаций
combo = ['/'.join(labels[row.astype(bool)]) if row.any() else '' for row in flags]
df_final_db_all_features['ТипАктивности'] = combo

# Если есть только органика — подставим "Органика"
if 'Органика' in df_final_db_all_features.columns:
    only_org = df_final_db_all_features['Органика'].fillna(0).astype('uint8').eq(1) & (flags.sum(axis=1) == 0)
    df_final_db_all_features.loc[only_org, 'ТипАктивности'] = 'Органика'

# Пустые — на "—"
df_final_db_all_features['ТипАктивности'] = df_final_db_all_features['ТипАктивности'].replace('', 'Органика')

# 1) Надёжно найдём колонку «Расход»
SPEND_CANDIDATES = ['Расход, ₽','Расход, руб','Расход, Р','Расход']
spend_col = next((c for c in SPEND_CANDIDATES if c in df_final_db_all_features.columns), None)
if spend_col is None:
    raise KeyError(f"Не найдена колонка расхода среди: {SPEND_CANDIDATES}")

# 2) Приведём «Расход» к числу
spend = pd.to_numeric(df_final_db_all_features[spend_col], errors='coerce')

# 3) Маска: было «Органика» и есть расход
is_organic = df_final_db_all_features['ТипАктивности'].astype('string').str.strip().eq('Органика')
has_spend  = spend.fillna(0).gt(0)      # строго > 0
# если вы хотели «> 0 ИЛИ not null», используйте так:
# has_spend = spend.notna() | spend.fillna(0).gt(0)
mask = is_organic & has_spend

# 4) Замена
df_final_db_all_features.loc[mask, 'ТипАктивности'] = 'Трафарет'

In [103]:
df_final_db_all_features['ТипАктивности'].value_counts()

ТипАктивности
Органика       2663720
Трафарет        412044
Вывод в топ         41
Name: count, dtype: int64

In [107]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2026-04-25']['Рекламные показы']

2921453   NaN
2921454   NaN
2921455   NaN
2921456   NaN
2921457   NaN
           ..
2971890   NaN
2971891   NaN
2971892   NaN
2971893   NaN
2971894   NaN
Name: Рекламные показы, Length: 50442, dtype: float32

In [83]:
print(list(df_final_db_all_features.columns))

['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталоге', 'Трафарет_Позиция в поиске и

In [84]:
df_final_db_all_features['Дата'].unique()

array(['2025-12-15', '2025-12-16', '2025-12-17', '2025-12-18',
       '2025-12-19', '2025-12-20', '2025-12-21', '2025-12-22',
       '2025-12-23', '2025-12-24', '2025-12-25', '2025-12-26',
       '2025-12-27', '2025-12-28', '2025-12-29', '2025-12-30',
       '2025-12-31', '2026-01-01', '2026-01-02', '2026-01-03',
       '2026-01-04', '2026-01-05', '2026-01-06', '2026-01-07',
       '2026-01-08', '2026-01-09', '2026-01-10', '2026-01-11',
       '2026-01-12', '2026-01-13', '2026-01-14', '2026-01-15',
       '2026-01-16', '2026-01-17', '2026-01-18', '2026-01-19',
       '2026-01-20', '2026-01-21', '2026-01-22', '2026-01-23',
       '2026-01-24', '2026-01-25', '2026-01-26', '2026-01-27',
       '2026-01-28', '2026-01-29', '2026-01-30', '2026-01-31',
       '2026-02-01', '2026-02-02', '2026-02-03', '2026-02-04',
       '2026-02-05'], dtype=object)

In [85]:
# === SQL СЦЕПКИ ОЗОН ===
sql = """
SELECT scepka.[id]
      ,scepka.[offer_id]
      ,scepka.[product_id]
      ,sku.fbo_sku as [Артикул OZ]
      ,scepka.[group_value] as [Текущая склейка]
      ,sku.[article]
      ,scepka.[updated_at] as [Дата Обновления]
  FROM [DBReport].[mp].[ozon_scepka] scepka
  JOIN [DBReport].[mp].[ozon_sku] sku 
  ON  scepka.[product_id] = sku.[product_id] 
  and sku.actual = 1
"""
df_links = pd.read_sql(sql, engine)
df_links['Артикул OZ'] = df_links['Артикул OZ'].astype(str)
df_links.to_excel(os.path.join(FOLDER_PATH, f"Склейки товаров\\OZ\\{df_links['Дата Обновления'].iloc[0].strftime('%d.%m.%Y')}_Склейка Товаров_OZ.xlsx"))

In [86]:
df_links['Текущая склейка'].unique()

array(['W8429001', '1716', '862', ..., '3599-2026-02-06_W8809454-37',
       '3002-2026-02-06_W7149078-38', '3596-2026-02-06_W8805284-36'],
      shape=(32629,), dtype=object)

In [87]:
print(list(df_final_db_all_features.columns))

['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталоге', 'Трафарет_Позиция в поиске и

In [88]:
df_final_db_all_features = pd.merge(df_final_db_all_features, df_links[["Артикул OZ", "Текущая склейка"]], how='left', on='Артикул OZ')

In [89]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '01.12.2025']['Расход, ₽'].sum()

np.float64(0.0)

In [90]:
df_final_db_all_features['Дата'] = pd.to_datetime(df_final_db_all_features['Дата'], format='%Y-%m-%d', errors='coerce').dt.strftime('%d.%m.%Y')

In [91]:
count = 0
for item in list(df_final_db_all_features.columns):
    print(f'{{"{item}", type {str(type(df_final_db_all_features[item].unique()[0]))}}}, ', end="")
    count +=1
    if count == 5:
        print("\n", end="")
        count = 0

{"Дата", type <class 'str'>}, {"Артикул", type <class 'str'>}, {"Артикул OZ", type <class 'str'>}, {"ТипАктивности", type <class 'str'>}, {"Наименование", type <class 'str'>}, 
{"Коллекция", type <class 'str'>}, {"Бренд", type <class 'str'>}, {"Сезон", type <class 'str'>}, {"Направление", type <class 'str'>}, {"Розничный отдел", type <class 'str'>}, 
{"Модель", type <class 'str'>}, {"Группа", type <class 'str'>}, {"Бизнес-группа", type <class 'str'>}, {"Техсегмент", type <class 'str'>}, {"Байер", type <class 'str'>}, 
{"Две последние коллекции", type <class 'str'>}, {"Основной артикул", type <class 'str'>}, {"Себестоимость с НДС", type <class 'numpy.float64'>}, {"Процент выкупа", type <class 'numpy.float64'>}, {"НДС", type <class 'numpy.float64'>}, 
{"Ответственный за группу", type <class 'str'>}, {"Группа для отчетов", type <class 'str'>}, {"SKU из объединенной карточки", type <class 'numpy.float64'>}, {"Ассоциированные заказы, руб", type <class 'numpy.float64'>}, {"Ассоциированные за

In [92]:
print(len(list(df_final_db_all_features.columns)))

132


In [93]:
df_final_db_all_features['ТипАктивности'].unique()

array(['Органика', 'Трафарет', 'Вывод в топ'], dtype=object)

In [94]:
df_final_db_all_features.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'ТипАктивности', 'Наименование',
       'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел',
       ...
       'Признак Артикула 2', 'Признак Артикула 3', 'Признак Артикула 4',
       'Признак Артикула 5', 'Признак Даты 1', 'Признак Даты 2',
       'Признак Даты 3', 'Признак Даты 4', 'Признак Даты 5',
       'Текущая склейка'],
      dtype='object', length=132)

In [95]:
# Сохранение финальной таблицы
import pyarrow as pa
import pyarrow.csv as csv
table = pa.Table.from_pandas(df_final_db_all_features)
csv.write_csv(table, os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon.csv"))
# df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon.csv"), index=False)

In [96]:
# del df_date_features
# gc.collect()

In [97]:
# Функция для обновления Excel-файла с циклом попыток
def update_and_save_excel(file_path, new_file_path):
    max_attempts = 10  # Максимальное количество попыток
    attempt = 0

    while attempt < max_attempts:
        attempt += 1
        print(f"Попытка {attempt} обновить файл '{os.path.basename(file_path)}'...")

        try:
            # Открываем Excel приложение
            excel = win32.Dispatch("Excel.Application")
            excel.DisplayAlerts = False  # Отключает предупреждения Excel

            try:
                # Открываем книгу
                workbook = excel.Workbooks.Open(file_path)

                # Выполняем обновление всех данных (эквивалентно "Обновить всё" в Excel)
                print("Выполняем обновление данных...")
                workbook.RefreshAll()
                excel.CalculateUntilAsyncQueriesDone()  # Дожидаемся завершения обновления

                # Сохраняем оригинальный файл в FOLDER_PATH_FOR_DB
                workbook.SaveAs(file_path)
                print(f"Файл успешно сохранен с оригинальным именем в '{os.path.dirname(file_path)}'.")

                # Сохраняем файл с новым именем в FOLDER_PATH_FEATURES
                workbook.SaveAs(new_file_path)
                print(f"Файл успешно сохранен как '{os.path.basename(new_file_path)}'.")

                return True  # Успешное завершение

            except Exception as e:
                print(f"Ошибка при обновлении или сохранении файла: {e}")
            finally:
                # Закрываем книгу и выходим из Excel
                if 'workbook' in locals():
                    workbook.Close(SaveChanges=False)
                excel.Quit()

        except Exception as e:
            print(f"Ошибка при работе с Excel: {e}")

        # Если произошла ошибка, ждем перед следующей попыткой
        if attempt < max_attempts:
            print(f"Пауза перед следующей попыткой ({attempt + 1}/{max_attempts})...")
            time.sleep(60)  # Пауза 5 секунд

    return False  # Все попытки завершились неудачно

In [98]:
# 19. Обновить файл "Показы и затраты ОЗ_2.0.xlsx"
try:
    print("Подготовка данных для ДБ завершена.")
    # input("Начать обновление файлов ДБ? Для подтверждения нажмите Enter...")
    print("Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...")
    start_time = time.time()  # Запускаем таймер

    # Путь к исходному файлу
    file_path_shows_expenses = os.path.join(FOLDER_PATH_FOR_DB, "Показы и затраты ОЗ_2.0.xlsx")

    if os.path.exists(file_path_shows_expenses):
        # Создаем новое имя файла с текущей датой без года
        current_month_day = time.strftime("%d.%m")  # Текущая дата в формате ДД.ММ
        new_file_name = f"Показы и затраты ОЗ_2.0 {current_month_day}.xlsx"
        new_file_path = os.path.join(FOLDER_PATH_FEATURES, new_file_name)

        # Путь для сохранения в дополнительную папку FOLDER_PATH_DUDL
        dudl_file_path = os.path.join(FOLDER_PATH_DUDL, new_file_name)

        # Удаляем старые файлы из FOLDER_PATH_DUDL
        try:
            if os.path.exists(FOLDER_PATH_DUDL):
                for filename in os.listdir(FOLDER_PATH_DUDL):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_DUDL, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_DUDL}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_DUDL}': {delete_error}")

        # Удаляем старые файлы из FOLDER_PATH_FEATURES
        try:
            if os.path.exists(FOLDER_PATH_FEATURES):
                for filename in os.listdir(FOLDER_PATH_FEATURES):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_FEATURES, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_FEATURES}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_FEATURES}': {delete_error}")

        # Пытаемся обновить и сохранить файл
        success = update_and_save_excel(file_path_shows_expenses, new_file_path)
        # success = True

        if not success:
            # Если все попытки неудачны, выводим сообщение пользователю
            while not success:
                input("Обновить Excel файл не получилось. Закройте все открытые файлы и нажмите любую кнопку для повторной попытки.")
                success = update_and_save_excel(file_path_shows_expenses, new_file_path)

            print("Файл успешно обновлен после повторной попытки.")

        # После успешного обновления копируем файл в папку FOLDER_PATH_DUDL
        if success:
            try:
                shutil.copy(new_file_path, dudl_file_path)
                print(f"Файл успешно скопирован в папку '{FOLDER_PATH_DUDL}'.")
            except Exception as copy_error:
                print(f"Ошибка при копировании файла в папку '{FOLDER_PATH_DUDL}': {copy_error}")

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Файл успешно обновлен и сохранен. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Показы и затраты ОЗ_2.0.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при обработке файла 'Показы и затраты ОЗ_2.0.xlsx': {e}")

Подготовка данных для ДБ завершена.
Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...
Файл 'Показы и затраты ОЗ_2.0 05.02.xlsx' удален из папки '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл 'Показы и затраты ОЗ_2.0 05.02.xlsx' удален из папки '\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям'.
Попытка 1 обновить файл 'Показы и затраты ОЗ_2.0.xlsx'...
Выполняем обновление данных...
Файл успешно сохранен с оригинальным именем в '\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям'.
Файл успешно сохранен как 'Показы и затраты ОЗ_2.0 06.02.xlsx'.
Файл успешно скопирован в папку '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл успешно обновлен и сохранен. Время выполнения: 0 часа(ов) 45 минут(ы) 39.86 секунд
